# Restaurant Turnover — V8 measured research

submission · submission · Team submission

The completed local round found **no qualifying V8 candidate**. Keep V7, whose user-reported portal RMSE is **12,024,316.5355221**. This notebook independently repeats all 16 fixed settings and 64 blend configurations, then freshly reproduces the scored V7 predictor. The unchanged V7 CSV/notebook are included as the fallback; uploading that CSV again does not create a new model.

All source is embedded. Only the four official data CSVs are required; previous project directories are not needed. No competitor code, predictions, weights or leaderboard-derived adjustments enter the predictor. All validation targets remain in INR scoring.

In [ ]:
import os, sys, json, base64, hashlib, subprocess, importlib.metadata, tempfile
from pathlib import Path
for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'LOKY_MAX_CPU_COUNT']:
    os.environ[key] = '2'
PACKAGES = {'numpy':'2.5.4', 'pandas':'3.0.6', 'scikit-learn':'1.9.1', 'scipy':'1.18.1', 'catboost':'1.2.10'}
needed = []
for package, version in PACKAGES.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        needed.append(f'{package}=={version}')
if needed:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *needed])
import numpy as np
import pandas as pd
print('Python:', sys.version.split()[0])
print({p: importlib.metadata.version(p) for p in PACKAGES})


## Inputs and storage

Set local MLA_DATA_DIR or upload the four original CSVs when prompted. Default execution is fresh. Optional Google Drive storage requires your authorization; free Colab resources vary. Set REUSE_CHECKPOINTS=True to resume matching checkpoints.

In [ ]:
USE_GOOGLE_DRIVE = False
REUSE_CHECKPOINTS = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['MLA_OUTPUT_DIR'] = '/content/drive/MyDrive/MLA_v8'
DATA_DIR = Path(os.environ.get('MLA_DATA_DIR', '.')).resolve()
OUTPUT_DIR = Path(os.environ.get('MLA_OUTPUT_DIR', '.')).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED = ['Train_dataset.csv', 'Test_dataset.csv', 'Data_Dictionary.csv', 'Sample_Solution.csv']
if not all((DATA_DIR / n).is_file() for n in REQUIRED):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place four official CSVs in DATA_DIR or set MLA_DATA_DIR.')
    files.upload()
    DATA_DIR = Path.cwd()
WORK_ROOT = OUTPUT_DIR / 'v8_resumable' if REUSE_CHECKPOINTS else Path(tempfile.mkdtemp(prefix='v8_fresh_', dir=OUTPUT_DIR))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('Research root:', WORK_ROOT)


In [ ]:
BUNDLE = json.loads('{"legacy_reference.py":"...","legacy_models.py":"IiIiVjc6IHR1cm5vdmVyIGJhbmRzLCBwYXJ0aWFsbHkgcG9vbGVkIHRpZXIgc3BlY2lhbGlzdHMgYW5kIG5lc3RlZCByZXNpZHVhbCBtb2RlbHMuIiIiCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIGNhdGJvb3N0IGltcG9ydCBDYXRCb29zdENsYXNzaWZpZXIsIENhdEJvb3N0UmVncmVzc29yCmZyb20gc2NpcHkuc3BlY2lhbCBpbXBvcnQgc29mdG1heApmcm9tIHNrbGVhcm4uY29tcG9zZSBpbXBvcnQgQ29sdW1uVHJhbnNmb3JtZXIKZnJvbSBza2xlYXJuLmltcHV0ZSBpbXBvcnQgU2ltcGxlSW1wdXRlcgpmcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBSaWRnZQpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBLRm9sZCwgR3JvdXBLRm9sZCwgU3RyYXRpZmllZEtGb2xkCmZyb20gc2tsZWFybi5waXBlbGluZSBpbXBvcnQgbWFrZV9waXBlbGluZQpmcm9tIHNrbGVhcm4ucHJlcHJvY2Vzc2luZyBpbXBvcnQgT25lSG90RW5jb2RlciwgU3RhbmRhcmRTY2FsZXIKaW1wb3J0IGxlZ2FjeV9yZWZlcmVuY2UgYXMgcmVmCgpSRVFVSVJFRCA9IFsnVHJhaW5fZGF0YXNldC5jc3YnLCAnVGVzdF9kYXRhc2V0LmNzdicsICdEYXRhX0RpY3Rpb25hcnkuY3N2JywgJ1NhbXBsZV9Tb2x1dGlvbi5jc3YnXQpTRUVEUyA9IFsxMDI0MiwgMjAyNDJdCkJBTkRTID0gWzIwLiwgNDAuLCA4MC5dClNQRUNTID0ge30KZm9yIGRlcHRoIGluIFszLCA1XToKICAgIGZvciBjYWxpYnJhdGUgaW4gW0ZhbHNlLCBUcnVlXToKICAgICAgICBTUEVDU1tmJ2JhbmRzX2R7ZGVwdGh9X2NhbHtpbnQoY2FsaWJyYXRlKX0nXSA9IHsKICAgICAgICAgICAgJ2ZhbWlseSc6ICdiYW5kcycsICdkZXB0aCc6IGRlcHRoLCAnY2FsaWJyYXRlJzogY2FsaWJyYXRlfQpmb3IgZGVwdGggaW4gWzQsIDZdOgogICAgZm9yIHNocmluayBpbiBbMTAwLCAzMDBdOgogICAgICAgIFNQRUNTW2YndGllcnNfZHtkZXB0aH1fc2hyaW5re3Nocmlua30nXSA9IHsKICAgICAgICAgICAgJ2ZhbWlseSc6ICd0aWVycycsICdkZXB0aCc6IGRlcHRoLCAnc2hyaW5rJzogc2hyaW5rfQpmb3IgYWxwaGEgaW4gWzEwMCwgMTAwMF06CiAgICBTUEVDU1tmJ3Jlc2lkdWFsX3JpZGdle2FscGhhfSddID0geydmYW1pbHknOiAncmVzaWR1YWwnLCAna2luZCc6ICdyaWRnZScsICdhbHBoYSc6IGFscGhhfQpmb3IgbG9zcyBpbiBbJ1JNU0UnLCAnSHViZXI6ZGVsdGE9MjAnXToKICAgIFNQRUNTWydyZXNpZHVhbF90cmVlXycgKyAoJ2wyJyBpZiBsb3NzID09ICdSTVNFJyBlbHNlICdoMjAnKV0gPSB7CiAgICAgICAgJ2ZhbWlseSc6ICdyZXNpZHVhbCcsICdraW5kJzogJ2NhdCcsICdsb3NzJzogbG9zc30KCgpkZWYgbG9hZF9kYXRhKGRpcmVjdG9yeSk6CiAgICBkaXJlY3RvcnkgPSBQYXRoKGRpcmVjdG9yeSkKICAgIGFzc2VydCBhbGwoKGRpcmVjdG9yeSAvIHApLmlzX2ZpbGUoKSBmb3IgcCBpbiBSRVFVSVJFRCksIGYnRm91ciBvZmZpY2lhbCBpbnB1dHMgcmVxdWlyZWQgaW4ge2RpcmVjdG9yeX0nCiAgICB0cmFpbiA9IHBkLnJlYWRfY3N2KGRpcmVjdG9yeSAvIFJFUVVJUkVEWzBdKS5yZW5hbWUoY29sdW1ucz17J0VuZG9yc2VkIEJ5JzogJ0VuZG9yZXNlZCBCeSd9KQogICAgdGVzdCA9IHBkLnJlYWRfY3N2KGRpcmVjdG9yeSAvIFJFUVVJUkVEWzFdKQogICAgc2FtcGxlID0gcGQucmVhZF9jc3YoZGlyZWN0b3J5IC8gUkVRVUlSRURbM10pCiAgICBwZC5yZWFkX2NzdihkaXJlY3RvcnkgLyBSRVFVSVJFRFsyXSkKICAgIGFzc2VydCBsZW4odGVzdCkgPT0gNTAwIGFuZCB0cmFpblsnUmVnaXN0cmF0aW9uIE51bWJlciddLmlzX3VuaXF1ZSBhbmQgdGVzdFsnUmVnaXN0cmF0aW9uIE51bWJlciddLmlzX3VuaXF1ZQogICAgYXNzZXJ0IHRyYWluLmRyb3AoY29sdW1ucz0nQW5udWFsIFR1cm5vdmVyJykuY29sdW1ucy50b2xpc3QoKSA9PSB0ZXN0LmNvbHVtbnMudG9saXN0KCkKICAgIGFzc2VydCBzZXQodHJhaW5bJ1JlZ2lzdHJhdGlvbiBOdW1iZXInXSkuaXNkaXNqb2ludCh0ZXN0WydSZWdpc3RyYXRpb24gTnVtYmVyJ10pCiAgICBhc3NlcnQgc2FtcGxlLmNvbHVtbnMudG9saXN0KCkgPT0gWydSZWdpc3RyYXRpb24gTnVtYmVyJywgJ0FubnVhbCBUdXJub3ZlciddCiAgICBhc3NlcnQgc2FtcGxlWydSZWdpc3RyYXRpb24gTnVtYmVyJ10uZXF1YWxzKHRlc3RbJ1JlZ2lzdHJhdGlvbiBOdW1iZXInXSkKICAgIHkgPSB0cmFpblsnQW5udWFsIFR1cm5vdmVyJ10udG9fbnVtcHkoZmxvYXQpCiAgICBhc3NlcnQgbnAuaXNmaW5pdGUoeSkuYWxsKCkgYW5kICh5ID4gMCkuYWxsKCkKICAgIHJldHVybiB0cmFpbiwgdGVzdCwgeQoKCmRlZiBmZWF0dXJlcyhyYXcpOgogICAgcmV0dXJuIHJlZi52NV9mZWF0dXJlcyhyYXcsICdjbGVhbicpCgoKZGVmIHNwbGl0X2luZGljZXMocmF3LCBzY2hlbWUpOgogICAgc2NoZW1lID0gc3RyKHNjaGVtZSkKICAgIGlmIHNjaGVtZSA9PSAnY2l0eSc6CiAgICAgICAgZ3JvdXBzID0gZmVhdHVyZXMocmF3KVsnQ2l0eSddCiAgICAgICAgcmVzdWx0ID0gbGlzdChHcm91cEtGb2xkKDUpLnNwbGl0KHJhdywgZ3JvdXBzPWdyb3VwcykpCiAgICAgICAgZm9yIGEsIGIgaW4gcmVzdWx0OgogICAgICAgICAgICBhc3NlcnQgc2V0KGdyb3Vwcy5pbG9jW2FdKS5pc2Rpc2pvaW50KGdyb3Vwcy5pbG9jW2JdKQogICAgZWxpZiBzY2hlbWUuc3RhcnRzd2l0aCgnc3RyYXQ6Jyk6CiAgICAgICAgc2VlZCA9IGludChzY2hlbWUuc3BsaXQoJzonKVsxXSkKICAgICAgICBiaW5zID0gbnAuZGlnaXRpemUocmF3WydBbm51YWwgVHVybm92ZXInXS50b19udW1weShmbG9hdCkgLyAxZTYsIEJBTkRTLCByaWdodD1UcnVlKQogICAgICAgIHJlc3VsdCA9IGxpc3QoU3RyYXRpZmllZEtGb2xkKDUsIHNodWZmbGU9VHJ1ZSwgcmFuZG9tX3N0YXRlPXNlZWQpLnNwbGl0KHJhdywgYmlucykpCiAgICBlbGlmIHNjaGVtZS5zdGFydHN3aXRoKCdpbm5lcjonKToKICAgICAgICBzZWVkID0gaW50KHNjaGVtZS5zcGxpdCgnOicpWzFdKQogICAgICAgIHJlc3VsdCA9IGxpc3QoS0ZvbGQoMywgc2h1ZmZsZT1UcnVlLCByYW5kb21fc3RhdGU9c2VlZCkuc3BsaXQocmF3KSkKICAgIGVsc2U6CiAgICAgICAgcmVzdWx0ID0gbGlzdChLRm9sZCg1LCBzaHVmZmxlPVRydWUsIHJhbmRvbV9zdGF0ZT1pbnQoc2NoZW1lKSkuc3BsaXQocmF3KSkKICAgIGNvdW50cyA9IG5wLnplcm9zKGxlbihyYXcpLCBpbnQpCiAgICBmb3IgYSwgYiBpbiByZXN1bHQ6CiAgICAgICAgYXNzZXJ0IG5vdCBzZXQoYSkuaW50ZXJzZWN0aW9uKGIpCiAgICAgICAgY291bnRzW2JdICs9IDEKICAgIGFzc2VydCAoY291bnRzID09IDEpLmFsbCgpCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIGNhdF9wYXJhbXMoc2VlZCwgZGVwdGgsIHRyZWVzLCBsMj0yMCk6CiAgICByZXR1cm4gZGljdChpdGVyYXRpb25zPXRyZWVzLCBkZXB0aD1kZXB0aCwgbGVhcm5pbmdfcmF0ZT0uMDQsIGwyX2xlYWZfcmVnPWwyLAogICAgICAgICAgICAgICAgYm9vdHN0cmFwX3R5cGU9J0Jlcm5vdWxsaScsIHN1YnNhbXBsZT0uNywgcmFuZG9tX3NlZWQ9c2VlZCwKICAgICAgICAgICAgICAgIHRocmVhZF9jb3VudD0yLCB2ZXJib3NlPUZhbHNlLCBhbGxvd193cml0aW5nX2ZpbGVzPUZhbHNlKQoKCmRlZiByZXNpZHVhbF9mZWF0dXJlcyhyYXcsIGJhc2VsaW5lKToKICAgIFggPSBmZWF0dXJlcyhyYXcpCiAgICBrZWVwID0gWydGYWNlYm9vayBQb3B1bGFyaXR5IFF1b3RpZW50JywgJ0luc3RhZ3JhbSBQb3B1bGFyaXR5IFF1b3RpZW50JywKICAgICAgICAgICAgJ1Jlc3R1cmFudCBUaWVyJywgJ0h5Z2llbmUgUmF0aW5nJywgJ1Jlc3RhdXJhbnQgWm9tYXRvIFJhdGluZycsCiAgICAgICAgICAgICdSZXN0YXVyYW50IENpdHkgVGllcicsICdPcGVuaW5nIFllYXInLCAnT3ZlcmFsbCBSZXN0YXVyYW50IFJhdGluZycsCiAgICAgICAgICAgICdDdWlzaW5lJywgJ1Jlc3RhdXJhbnQgVHlwZScsICdSZXN0YXVyYW50IExvY2F0aW9uJ10KICAgIFogPSBYW2tlZXBdLmNvcHkoKQogICAgWlsnQmFzZSBwcmVkaWN0aW9uJ10gPSBucC5hc2FycmF5KGJhc2VsaW5lKSAvIDFlNgogICAgWlsnU29jaWFsIG1lYW4nXSA9IFpbWydGYWNlYm9vayBQb3B1bGFyaXR5IFF1b3RpZW50JywgJ0luc3RhZ3JhbSBQb3B1bGFyaXR5IFF1b3RpZW50J11dLm1lYW4oYXhpcz0xKQogICAgWlsnU29jaWFsIGh5Z2llbmUnXSA9IFpbJ1NvY2lhbCBtZWFuJ10gKiBaWydIeWdpZW5lIFJhdGluZyddIC8gMTAwCiAgICBaWydUaWVyMSBiYXNlJ10gPSBaWydCYXNlIHByZWRpY3Rpb24nXSAqIChaWydSZXN0dXJhbnQgVGllciddID09IDEpCiAgICBaWydIeWdpZW5lIGJhc2UnXSA9IFpbJ0Jhc2UgcHJlZGljdGlvbiddICogKFpbJ0h5Z2llbmUgUmF0aW5nJ10gLSA1KSAvIDUKICAgIHJldHVybiBaCgoKZGVmIGZpdF9jb21wb25lbnQocmF3LCB5LCBzcGVjLCBzZWVkLCBpbm5lcl9iYXNlbGluZT1Ob25lKToKICAgIFggPSBmZWF0dXJlcyhyYXcpCiAgICB0YXJnZXQgPSBucC5hc2FycmF5KHksIGZsb2F0KSAvIDFlNgogICAgY2F0cyA9IFguc2VsZWN0X2R0eXBlcyhpbmNsdWRlPVsnb2JqZWN0JywgJ3N0cmluZyddKS5jb2x1bW5zLnRvbGlzdCgpCiAgICBpbmZvID0geydzZWVkJzogc2VlZCwgJ3RyYWluX3Jvd3MnOiBsZW4ocmF3KSwgJ2ZhbWlseSc6IHNwZWNbJ2ZhbWlseSddfQogICAgaWYgc3BlY1snZmFtaWx5J10gPT0gJ2JhbmRzJzoKICAgICAgICBsYWJlbHMgPSBucC5kaWdpdGl6ZSh0YXJnZXQsIEJBTkRTLCByaWdodD1UcnVlKQogICAgICAgIGNvdW50cyA9IG5wLmJpbmNvdW50KGxhYmVscywgbWlubGVuZ3RoPTQpCiAgICAgICAgYXNzZXJ0IChjb3VudHMgPj0gMykuYWxsKCkKICAgICAgICBjZW50ZXJzID0gbnAuYXJyYXkoW3RhcmdldFtsYWJlbHMgPT0ga10ubWVhbigpIGZvciBrIGluIHJhbmdlKDQpXSkKICAgICAgICB0ZW1wZXJhdHVyZSA9IDEuCiAgICAgICAgaWYgc3BlY1snY2FsaWJyYXRlJ106CiAgICAgICAgICAgIHByb2IgPSBucC56ZXJvcygobGVuKFgpLCA0KSkKICAgICAgICAgICAgZm9yIGYsIChhLCBiKSBpbiBlbnVtZXJhdGUoU3RyYXRpZmllZEtGb2xkKDMsIHNodWZmbGU9VHJ1ZSwgcmFuZG9tX3N0YXRlPXNlZWQgKyA0MTEpLnNwbGl0KFgsIGxhYmVscykpOgogICAgICAgICAgICAgICAgbW9kZWwgPSBDYXRCb29zdENsYXNzaWZpZXIoKipjYXRfcGFyYW1zKHNlZWQgKyAxMDAgKyBmLCBzcGVjWydkZXB0aCddLCA4MDApLCBsb3NzX2Z1bmN0aW9uPSdNdWx0aUNsYXNzJykKICAgICAgICAgICAgICAgIG1vZGVsLmZpdChYLmlsb2NbYV0sIGxhYmVsc1thXSwgY2F0X2ZlYXR1cmVzPWNhdHMpCiAgICAgICAgICAgICAgICBhc3NlcnQgbnAuYXJyYXlfZXF1YWwobW9kZWwuY2xhc3Nlc18sIG5wLmFyYW5nZSg0KSkKICAgICAgICAgICAgICAgIHByb2JbYl0gPSBtb2RlbC5wcmVkaWN0X3Byb2JhKFguaWxvY1tiXSkKICAgICAgICAgICAgbG9nX3Byb2IgPSBucC5sb2cobnAuY2xpcChwcm9iLCAxZS0xMiwgMSkpCiAgICAgICAgICAgIHRlbXBlcmF0dXJlcyA9IFsuNywgMS4sIDEuNCwgMi5dCiAgICAgICAgICAgIGxvc3NlcyA9IFstbnAubWVhbihucC5sb2cobnAuY2xpcChzb2Z0bWF4KGxvZ19wcm9iIC8gdCwgYXhpcz0xKVtucC5hcmFuZ2UobGVuKFgpKSwgbGFiZWxzXSwgMWUtMTIsIDEpKSkgZm9yIHQgaW4gdGVtcGVyYXR1cmVzXQogICAgICAgICAgICB0ZW1wZXJhdHVyZSA9IHRlbXBlcmF0dXJlc1tpbnQobnAuYXJnbWluKGxvc3NlcykpXQogICAgICAgICAgICBpbmZvLnVwZGF0ZSh0ZW1wZXJhdHVyZV9jYW5kaWRhdGVzPXRlbXBlcmF0dXJlcywgaW5uZXJfb29mX2xvZ2xvc3M9bG9zc2VzKQogICAgICAgIG1vZGVsID0gQ2F0Qm9vc3RDbGFzc2lmaWVyKCoqY2F0X3BhcmFtcyhzZWVkLCBzcGVjWydkZXB0aCddLCA4MDApLCBsb3NzX2Z1bmN0aW9uPSdNdWx0aUNsYXNzJykKICAgICAgICBtb2RlbC5maXQoWCwgbGFiZWxzLCBjYXRfZmVhdHVyZXM9Y2F0cykKICAgICAgICBhc3NlcnQgbnAuYXJyYXlfZXF1YWwobW9kZWwuY2xhc3Nlc18sIG5wLmFyYW5nZSg0KSkKICAgICAgICBpbmZvLnVwZGF0ZShiYW5kX2NvdW50cz1jb3VudHMudG9saXN0KCksIGJhbmRfY2VudGVyc19taWxsaW9ucz1jZW50ZXJzLnRvbGlzdCgpLCB0ZW1wZXJhdHVyZT10ZW1wZXJhdHVyZSkKICAgICAgICByZXR1cm4geydmYW1pbHknOiAnYmFuZHMnLCAnbW9kZWwnOiBtb2RlbCwgJ2NlbnRlcnMnOiBjZW50ZXJzLCAndGVtcGVyYXR1cmUnOiB0ZW1wZXJhdHVyZSwgJ2luZm8nOiBpbmZvfQogICAgaWYgc3BlY1snZmFtaWx5J10gPT0gJ3RpZXJzJzoKICAgICAgICBnbG9iYWxfbW9kZWwgPSBDYXRCb29zdFJlZ3Jlc3NvcigqKmNhdF9wYXJhbXMoc2VlZCwgNSwgODUwLCAxMCksIGxvc3NfZnVuY3Rpb249J0h1YmVyOmRlbHRhPTQwJykKICAgICAgICBnbG9iYWxfbW9kZWwuZml0KFgsIHRhcmdldCwgY2F0X2ZlYXR1cmVzPWNhdHMpCiAgICAgICAgbG9jYWwgPSB7fQogICAgICAgIGZvciB0aWVyIGluIFsxLiwgMi5dOgogICAgICAgICAgICBtYXNrID0gWFsnUmVzdHVyYW50IFRpZXInXS50b19udW1weSgpID09IHRpZXIKICAgICAgICAgICAgbiA9IGludChtYXNrLnN1bSgpKQogICAgICAgICAgICBpZiBuIDwgMTIwOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgbW9kZWwgPSBDYXRCb29zdFJlZ3Jlc3NvcigqKmNhdF9wYXJhbXMoc2VlZCArIGludCh0aWVyKSAqIDcxLCBzcGVjWydkZXB0aCddLCA2NTAsIDMwKSwgbG9zc19mdW5jdGlvbj0nSHViZXI6ZGVsdGE9NDAnKQogICAgICAgICAgICBtb2RlbC5maXQoWC5sb2NbbWFza10sIHRhcmdldFttYXNrXSwgY2F0X2ZlYXR1cmVzPWNhdHMpCiAgICAgICAgICAgIGxvY2FsW3RpZXJdID0geydtb2RlbCc6IG1vZGVsLCAnd2VpZ2h0JzogbiAvIChuICsgc3BlY1snc2hyaW5rJ10pLCAncm93cyc6IG59CiAgICAgICAgaW5mb1snbG9jYWxfdGllcnMnXSA9IHtzdHIodCk6IHsnd2VpZ2h0JzogdmFsdWVbJ3dlaWdodCddLCAncm93cyc6IHZhbHVlWydyb3dzJ119IGZvciB0LCB2YWx1ZSBpbiBsb2NhbC5pdGVtcygpfQogICAgICAgIHJldHVybiB7J2ZhbWlseSc6ICd0aWVycycsICdnbG9iYWwnOiBnbG9iYWxfbW9kZWwsICdsb2NhbCc6IGxvY2FsLCAnaW5mbyc6IGluZm99CiAgICBhc3NlcnQgc3BlY1snZmFtaWx5J10gPT0gJ3Jlc2lkdWFsJyBhbmQgaW5uZXJfYmFzZWxpbmUgaXMgbm90IE5vbmUKICAgIGlubmVyX2Jhc2VsaW5lID0gbnAuYXNhcnJheShpbm5lcl9iYXNlbGluZSwgZmxvYXQpCiAgICBhc3NlcnQgaW5uZXJfYmFzZWxpbmUuc2hhcGUgPT0gdGFyZ2V0LnNoYXBlIGFuZCBucC5pc2Zpbml0ZShpbm5lcl9iYXNlbGluZSkuYWxsKCkKICAgIFogPSByZXNpZHVhbF9mZWF0dXJlcyhyYXcsIGlubmVyX2Jhc2VsaW5lKQogICAgcmVzaWR1YWwgPSB0YXJnZXQgLSBpbm5lcl9iYXNlbGluZSAvIDFlNgogICAgaWYgc3BlY1sna2luZCddID09ICdyaWRnZSc6CiAgICAgICAgY2F0ZWdvcmljYWwgPSBaLnNlbGVjdF9kdHlwZXMoaW5jbHVkZT1bJ29iamVjdCcsICdzdHJpbmcnXSkuY29sdW1ucy50b2xpc3QoKQogICAgICAgIG51bWVyaWMgPSBbYyBmb3IgYyBpbiBaIGlmIGMgbm90IGluIGNhdGVnb3JpY2FsXQogICAgICAgIHByZSA9IENvbHVtblRyYW5zZm9ybWVyKFsKICAgICAgICAgICAgKCdudW0nLCBtYWtlX3BpcGVsaW5lKFNpbXBsZUltcHV0ZXIoc3RyYXRlZ3k9J21lZGlhbicsIGFkZF9pbmRpY2F0b3I9VHJ1ZSksIFN0YW5kYXJkU2NhbGVyKCkpLCBudW1lcmljKSwKICAgICAgICAgICAgKCdjYXQnLCBPbmVIb3RFbmNvZGVyKGhhbmRsZV91bmtub3duPSdpZ25vcmUnLCBtaW5fZnJlcXVlbmN5PTEwLCBzcGFyc2Vfb3V0cHV0PUZhbHNlKSwgY2F0ZWdvcmljYWwpXSkKICAgICAgICBtb2RlbCA9IG1ha2VfcGlwZWxpbmUocHJlLCBSaWRnZShhbHBoYT1zcGVjWydhbHBoYSddKSkKICAgICAgICBtb2RlbC5maXQoWiwgcmVzaWR1YWwpCiAgICBlbHNlOgogICAgICAgIG1vZGVsID0gQ2F0Qm9vc3RSZWdyZXNzb3IoKipjYXRfcGFyYW1zKHNlZWQsIDIsIDMwMCwgNTApLCBsb3NzX2Z1bmN0aW9uPXNwZWNbJ2xvc3MnXSkKICAgICAgICBtb2RlbC5maXQoWiwgcmVzaWR1YWwsIGNhdF9mZWF0dXJlcz1aLnNlbGVjdF9kdHlwZXMoaW5jbHVkZT1bJ29iamVjdCcsICdzdHJpbmcnXSkuY29sdW1ucy50b2xpc3QoKSkKICAgIGluZm8udXBkYXRlKGlubmVyX2Jhc2VsaW5lPSd0aHJlZS1mb2xkIHY1IE9PRiB3aXRoaW4gdGhpcyBvdXRlciB0cmFpbmluZyBmb2xkJywgbWVhbl9pbm5lcl9yZXNpZHVhbF9taWxsaW9ucz1mbG9hdChyZXNpZHVhbC5tZWFuKCkpKQogICAgcmV0dXJuIHsnZmFtaWx5JzogJ3Jlc2lkdWFsJywgJ21vZGVsJzogbW9kZWwsICdpbmZvJzogaW5mb30KCgpkZWYgcHJlZGljdF9jb21wb25lbnQoZml0dGVkLCByYXcsIGJhc2VsaW5lPU5vbmUpOgogICAgaWYgZml0dGVkWydmYW1pbHknXSA9PSAnYmFuZHMnOgogICAgICAgIHAgPSBmaXR0ZWRbJ21vZGVsJ10ucHJlZGljdF9wcm9iYShmZWF0dXJlcyhyYXcpKQogICAgICAgIHAgPSBzb2Z0bWF4KG5wLmxvZyhucC5jbGlwKHAsIDFlLTEyLCAxKSkgLyBmaXR0ZWRbJ3RlbXBlcmF0dXJlJ10sIGF4aXM9MSkKICAgICAgICBwcmVkaWN0aW9uID0gcCBAIGZpdHRlZFsnY2VudGVycyddICogMWU2CiAgICBlbGlmIGZpdHRlZFsnZmFtaWx5J10gPT0gJ3RpZXJzJzoKICAgICAgICBYID0gZmVhdHVyZXMocmF3KQogICAgICAgIHByZWRpY3Rpb24gPSBmaXR0ZWRbJ2dsb2JhbCddLnByZWRpY3QoWCkKICAgICAgICBmb3IgdGllciwgbG9jYWwgaW4gZml0dGVkWydsb2NhbCddLml0ZW1zKCk6CiAgICAgICAgICAgIG1hc2sgPSBYWydSZXN0dXJhbnQgVGllciddLnRvX251bXB5KCkgPT0gdGllcgogICAgICAgICAgICBpZiBtYXNrLmFueSgpOgogICAgICAgICAgICAgICAgcHJlZGljdGlvblttYXNrXSA9ICgxIC0gbG9jYWxbJ3dlaWdodCddKSAqIHByZWRpY3Rpb25bbWFza10gKyBsb2NhbFsnd2VpZ2h0J10gKiBsb2NhbFsnbW9kZWwnXS5wcmVkaWN0KFgubG9jW21hc2tdKQogICAgICAgIHByZWRpY3Rpb24gKj0gMWU2CiAgICBlbHNlOgogICAgICAgIGFzc2VydCBiYXNlbGluZSBpcyBub3QgTm9uZSBhbmQgbGVuKGJhc2VsaW5lKSA9PSBsZW4ocmF3KQogICAgICAgIHByZWRpY3Rpb24gPSBucC5hc2FycmF5KGJhc2VsaW5lKSArIGZpdHRlZFsnbW9kZWwnXS5wcmVkaWN0KHJlc2lkdWFsX2ZlYXR1cmVzKHJhdywgYmFzZWxpbmUpKSAqIDFlNgogICAgcHJlZGljdGlvbiA9IG5wLm1heGltdW0ocHJlZGljdGlvbiwgMCkKICAgIGFzc2VydCBwcmVkaWN0aW9uLnNoYXBlID09IChsZW4ocmF3KSwpIGFuZCBucC5pc2Zpbml0ZShwcmVkaWN0aW9uKS5hbGwoKQogICAgcmV0dXJuIHByZWRpY3Rpb24KCgpkZWYgZXZhbHVhdGVfY29tcG9uZW50KHJhdywgeSwgc3BlYywgc2NoZW1lLCBiYXNlbGluZSwgaW5uZXJfcHJvdmlkZXI9Tm9uZSwgcHJvZ3Jlc3M9Tm9uZSk6CiAgICBpbmRpdmlkdWFsID0gbnAuemVyb3MoKGxlbih5KSwgbGVuKFNFRURTKSkpCiAgICBkaWFnbm9zdGljcyA9IFtdCiAgICBmb3IgZm9sZCwgKGEsIGIpIGluIGVudW1lcmF0ZShzcGxpdF9pbmRpY2VzKHJhdywgc2NoZW1lKSk6CiAgICAgICAgaW5uZXIgPSBpbm5lcl9wcm92aWRlcihhLCBmb2xkKSBpZiBzcGVjWydmYW1pbHknXSA9PSAncmVzaWR1YWwnIGVsc2UgTm9uZQogICAgICAgIGZvciBqLCBzZWVkIGluIGVudW1lcmF0ZShTRUVEUyk6CiAgICAgICAgICAgIGZpdHRlZCA9IGZpdF9jb21wb25lbnQocmF3Lmlsb2NbYV0sIHlbYV0sIHNwZWMsIHNlZWQgKyBmb2xkLCBpbm5lcikKICAgICAgICAgICAgaW5kaXZpZHVhbFtiLCBqXSA9IHByZWRpY3RfY29tcG9uZW50KGZpdHRlZCwgcmF3Lmlsb2NbYl0sIGJhc2VsaW5lW2JdKQogICAgICAgICAgICBkaWFnbm9zdGljcy5hcHBlbmQoeydmb2xkJzogZm9sZCwgKipmaXR0ZWRbJ2luZm8nXX0pCiAgICAgICAgaWYgcHJvZ3Jlc3M6CiAgICAgICAgICAgIHByb2dyZXNzKGYnZm9sZCB7Zm9sZCArIDF9LzUgY29tcGxldGUnKQogICAgcmV0dXJuIGluZGl2aWR1YWwubWVhbihheGlzPTEpLCBpbmRpdmlkdWFsLCBkaWFnbm9zdGljcwoKCmRlZiB2NV9mb2xkX3ByZWRpY3Rpb24odHJhaW4sIHksIHZhbGlkLCBjb250cmFjdCwgZm9sZCk6CiAgICAiIiJFeGFjdCBsZWdhY3kgb3V0ZXItZm9sZCBzZWVkIHNjaGVkdWxlOyBubyB2YWxpZGF0aW9uIGxhYmVscyBlbnRlciBmaXR0aW5nLiIiIgogICAgbiA9IGxlbih0cmFpbikKICAgIGEgPSBucC5hcmFuZ2UobikKICAgIGIgPSBucC5hcmFuZ2UobiwgbiArIGxlbih2YWxpZCkpCiAgICBqb2luZWRfeSA9IG5wLnJfW25wLmFzYXJyYXkoeSksIG5wLmZ1bGwobGVuKHZhbGlkKSwgbnAubmFuKV0KICAgIHByaW9yID0gbnAuemVyb3MobGVuKHZhbGlkKSkKICAgIGZvciBuYW1lLCB3ZWlnaHQgaW4gY29udHJhY3RbJ3YyX3dlaWdodHMnXS5pdGVtcygpOgogICAgICAgIHNwZWMgPSBjb250cmFjdFsndjJfY29uZmlncyddW25hbWVdCiAgICAgICAgWCA9IHBkLmNvbmNhdChbcmVmLmZlYXR1cmVzKHRyYWluLCBzcGVjWyd2YXJpYW50J10pLCByZWYuZmVhdHVyZXModmFsaWQsIHNwZWNbJ3ZhcmlhbnQnXSldLCBpZ25vcmVfaW5kZXg9VHJ1ZSkKICAgICAgICBwLCBfID0gcmVmLmZpdF9wcmVkaWN0KFgsIGpvaW5lZF95LCBhLCBiLCBzcGVjLCA0MyArIGZvbGQsIHNwZWMuZ2V0KCdpdGVyYXRpb25zJykpCiAgICAgICAgcHJpb3IgKz0gLjUgKiB3ZWlnaHQgKiBwCiAgICBmb3Igc2VlZCBpbiBjb250cmFjdFsndjNfc2VlZHMnXToKICAgICAgICBtb2RlbCA9IHJlZi5jYXRfZml0KHJlZi5mZWF0dXJlcyh0cmFpbiwgJ2Jhc2UnKSwgbnAuYXNhcnJheSh5KSwgY29udHJhY3RbJ3YzX3BhcmFtZXRlcnMnXSwgc2VlZCArIGZvbGQpCiAgICAgICAgcHJpb3IgKz0gLjUgKiBucC5tYXhpbXVtKG1vZGVsLnByZWRpY3QocmVmLmZlYXR1cmVzKHZhbGlkLCAnYmFzZScpKSwgMCkgLyBsZW4oY29udHJhY3RbJ3YzX3NlZWRzJ10pCiAgICBwID0gY29udHJhY3RbJ3YzX3dlaWdodCddICogcHJpb3IKICAgIGZvciBuYW1lLCBjb21wb25lbnQgaW4gY29udHJhY3RbJ2NvbXBvbmVudHMnXS5pdGVtcygpOgogICAgICAgIGZvciBzZWVkIGluIGNvbnRyYWN0WydzZWVkcyddOgogICAgICAgICAgICBtb2RlbCA9IHJlZi5maXRfdjUodHJhaW4sIG5wLmFzYXJyYXkoeSksIGNvbXBvbmVudFsnc3BlYyddLCBzZWVkICsgZm9sZCkKICAgICAgICAgICAgcCArPSBjb21wb25lbnRbJ3dlaWdodCddICogcmVmLnByZWRpY3RfdjUobW9kZWwsIHZhbGlkKSAvIGxlbihjb250cmFjdFsnc2VlZHMnXSkKICAgIHJldHVybiBwCgoKZGVmIGlubmVyX3Y1X29vZihyYXcsIHksIGNvbnRyYWN0LCBzZWVkPTczMSwgcHJvZ3Jlc3M9Tm9uZSk6CiAgICBwcmVkaWN0aW9uID0gbnAuemVyb3MobGVuKHJhdykpCiAgICBmb3IgZiwgKGEsIGIpIGluIGVudW1lcmF0ZShzcGxpdF9pbmRpY2VzKHJhdywgZidpbm5lcjp7c2VlZH0nKSk6CiAgICAgICAgcHJlZGljdGlvbltiXSA9IHY1X2ZvbGRfcHJlZGljdGlvbihyYXcuaWxvY1thXSwgbnAuYXNhcnJheSh5KVthXSwgcmF3Lmlsb2NbYl0sIGNvbnRyYWN0LCBmKQogICAgICAgIGlmIHByb2dyZXNzOgogICAgICAgICAgICBwcm9ncmVzcyhmJ2lubmVyIGZvbGQge2YgKyAxfS8zIGNvbXBsZXRlJykKICAgIHJldHVybiBwcmVkaWN0aW9uCgoKZGVmIGZpdF9zdWJtaXNzaW9uKHRyYWluLCB0ZXN0LCB5LCBjb250cmFjdCwgY2hvaWNlLCBpbm5lcl9iYXNlbGluZT1Ob25lKToKICAgICMgRnJlc2ggYmFzZWxpbmUsIGluY2x1ZGluZyBpbmRlcGVuZGVudCB2ZXJpZmljYXRpb24gYWdhaW5zdCB0aGUgc2NvcmVkIHY1IGZpbGUgb3V0c2lkZSB0aGlzIGZ1bmN0aW9uLgogICAgXywgYmFzZWxpbmUsIGxlZ2FjeSA9IHJlZi5mdWxsX2ZpdCh0cmFpbiwgdGVzdCwgeSwgY29udHJhY3QpCiAgICBmaXR0ZWQgPSB7J2Nob2ljZSc6IGNob2ljZSwgJ2xlZ2FjeSc6IGxlZ2FjeSwgJ2NvbnRyYWN0JzogY29udHJhY3QsICdtb2RlbHMnOiBbXX0KICAgIHByZWRpY3Rpb24gPSAoMSAtIGNob2ljZVsnd2VpZ2h0J10pICogYmFzZWxpbmUKICAgIGZvciBzZWVkIGluIFNFRURTOgogICAgICAgIG1vZGVsID0gZml0X2NvbXBvbmVudCh0cmFpbiwgeSwgU1BFQ1NbY2hvaWNlWyduYW1lJ11dLCBzZWVkLCBpbm5lcl9iYXNlbGluZSkKICAgICAgICBmaXR0ZWRbJ21vZGVscyddLmFwcGVuZChtb2RlbCkKICAgICAgICBwcmVkaWN0aW9uICs9IGNob2ljZVsnd2VpZ2h0J10gKiBwcmVkaWN0X2NvbXBvbmVudChtb2RlbCwgdGVzdCwgYmFzZWxpbmUpIC8gbGVuKFNFRURTKQogICAgcmV0dXJuIHByZWRpY3Rpb24sIGJhc2VsaW5lLCBmaXR0ZWQKCgpkZWYgcHJlZGljdF9zdWJtaXNzaW9uKGZpdHRlZCwgcmF3KToKICAgIGJhc2VsaW5lID0gcmVmLnByZWRpY3RfZmluYWwocmF3LCBmaXR0ZWRbJ2xlZ2FjeSddLCBmaXR0ZWRbJ2NvbnRyYWN0J10pCiAgICBjaG9pY2UgPSBmaXR0ZWRbJ2Nob2ljZSddCiAgICBjb21wb25lbnQgPSBucC5tZWFuKFtwcmVkaWN0X2NvbXBvbmVudChtLCByYXcsIGJhc2VsaW5lKSBmb3IgbSBpbiBmaXR0ZWRbJ21vZGVscyddXSwgYXhpcz0wKQogICAgcmV0dXJuICgxIC0gY2hvaWNlWyd3ZWlnaHQnXSkgKiBiYXNlbGluZSArIGNob2ljZVsnd2VpZ2h0J10gKiBjb21wb25lbnQK","baseline.py":"IiIiRXhhY3QgZnJvemVuIFY3IFRyaWFsIDEgZml0dGluZywgaW5kZXBlbmRlbnQgb2YgcHJldmlvdXMgcHJvamVjdCBkaXJlY3Rvcmllcy4iIiIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBsZWdhY3lfcmVmZXJlbmNlIGFzIHJlZgppbXBvcnQgbGVnYWN5X21vZGVscyBhcyBsZWdhY3kKClBPUlRBTF9STVNFID0gMTIwMjQzMTYuNTM1NTIyMQpDSE9JQ0UgPSB7J25hbWUnOiAncmVzaWR1YWxfdHJlZV9sMicsICdmYW1pbHknOiAncmVzaWR1YWwnLCAnd2VpZ2h0JzogLjc1fQoKCmRlZiBmb2xkX3ByZWRpY3Rpb24odHJhaW4sIHksIHZhbGlkLCBjb250cmFjdCwgZm9sZCwgcHJvZ3Jlc3M9Tm9uZSk6CiAgICBiID0gbGVnYWN5LnY1X2ZvbGRfcHJlZGljdGlvbih0cmFpbiwgeSwgdmFsaWQsIGNvbnRyYWN0LCBmb2xkKQogICAgaWYgcHJvZ3Jlc3M6CiAgICAgICAgcHJvZ3Jlc3MoJ3Y1IG91dGVyIGZpdCBjb21wbGV0ZTsgZ2VuZXJhdGluZyBuZXN0ZWQgcmVzaWR1YWwgaW5wdXRzJykKICAgIGlubmVyID0gbGVnYWN5LmlubmVyX3Y1X29vZih0cmFpbiwgeSwgY29udHJhY3QsIDczMSArIGZvbGQsIHByb2dyZXNzKQogICAgbW9kZWxzID0gW2xlZ2FjeS5maXRfY29tcG9uZW50KHRyYWluLCB5LCBsZWdhY3kuU1BFQ1NbQ0hPSUNFWyduYW1lJ11dLCBzZWVkICsgZm9sZCwgaW5uZXIpCiAgICAgICAgICAgICAgZm9yIHNlZWQgaW4gbGVnYWN5LlNFRURTXQogICAgcCA9IC4yNSAqIGIgKyAuNzUgKiBucC5tZWFuKFtsZWdhY3kucHJlZGljdF9jb21wb25lbnQobSwgdmFsaWQsIGIpIGZvciBtIGluIG1vZGVsc10sIGF4aXM9MCkKICAgIHJldHVybiBwCgoKZGVmIGZpdF9mdWxsKHRyYWluLCB0ZXN0LCB5LCBjb250cmFjdCwgcHJvZ3Jlc3M9Tm9uZSk6CiAgICBpbm5lciA9IGxlZ2FjeS5pbm5lcl92NV9vb2YodHJhaW4sIHksIGNvbnRyYWN0LCA3MzEsIHByb2dyZXNzKQogICAgcCwgYiwgZml0dGVkID0gbGVnYWN5LmZpdF9zdWJtaXNzaW9uKHRyYWluLCB0ZXN0LCB5LCBjb250cmFjdCwgQ0hPSUNFLCBpbm5lcikKICAgIHJldHVybiBwLCBmaXR0ZWQKCgpkZWYgcHJlZGljdChmaXR0ZWQsIHJhdyk6CiAgICByZXR1cm4gbGVnYWN5LnByZWRpY3Rfc3VibWlzc2lvbihmaXR0ZWQsIHJhdykK","modeling.py":"IiIiT3JpZ2luYWwgVjggZXhwZXJpbWVudHM6IG5hdGl2ZSBIR0IsIGNyb3NzLWZpdHRlZCBzdGF0aXN0aWNzIGFuZCBSaWRnZS4iIiIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBza2xlYXJuLmNvbXBvc2UgaW1wb3J0IENvbHVtblRyYW5zZm9ybWVyCmZyb20gc2tsZWFybi5lbnNlbWJsZSBpbXBvcnQgSGlzdEdyYWRpZW50Qm9vc3RpbmdSZWdyZXNzb3IKZnJvbSBza2xlYXJuLmltcHV0ZSBpbXBvcnQgU2ltcGxlSW1wdXRlcgpmcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBSaWRnZQpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBLRm9sZApmcm9tIHNrbGVhcm4ucGlwZWxpbmUgaW1wb3J0IG1ha2VfcGlwZWxpbmUKZnJvbSBza2xlYXJuLnByZXByb2Nlc3NpbmcgaW1wb3J0IE9uZUhvdEVuY29kZXIsIFN0YW5kYXJkU2NhbGVyLCBUYXJnZXRFbmNvZGVyCmltcG9ydCBsZWdhY3lfbW9kZWxzIGFzIGxlZ2FjeQoKU0VFRCA9IDEwMjQyCldFSUdIVFMgPSBbLjI1LCAuNSwgLjc1LCAxLl0KU1BFQ1MgPSB7fQpmb3IgdGFyZ2V0IGluIFsncmF3JywgJ3NxcnQnXToKICAgIGZvciBsZWF2ZXMgaW4gWzcsIDE1XToKICAgICAgICBTUEVDU1tmJ25hdGl2ZV97dGFyZ2V0fV97bGVhdmVzfSddID0gZGljdChmYW1pbHk9J25hdGl2ZScsIGdyb3VwPSdoZ2InLCB0YXJnZXQ9dGFyZ2V0LCBsZWF2ZXM9bGVhdmVzKQogICAgZm9yIHNtb290aCBpbiBbNTAsIDIwMF06CiAgICAgICAgU1BFQ1NbZidlbmNvZGVkX3t0YXJnZXR9X3tzbW9vdGh9J10gPSBkaWN0KGZhbWlseT0nZW5jb2RlZCcsIGdyb3VwPSdoZ2InLCB0YXJnZXQ9dGFyZ2V0LCBsZWF2ZXM9MTUsIHNtb290aD1zbW9vdGgpCiAgICBmb3IgYWxwaGEgaW4gWzEwLCAxMDBdOgogICAgICAgIGZvciBzdHJ1Y3R1cmVkIGluIFtGYWxzZSwgVHJ1ZV06CiAgICAgICAgICAgIFNQRUNTW2YncmlkZ2Vfe3RhcmdldH1fe2FscGhhfV97InN0cnVjdHVyZWQiIGlmIHN0cnVjdHVyZWQgZWxzZSAiYmFzZSJ9J10gPSBkaWN0KAogICAgICAgICAgICAgICAgZmFtaWx5PSdyaWRnZScsIGdyb3VwPSdyaWRnZScsIHRhcmdldD10YXJnZXQsIGFscGhhPWFscGhhLCBzdHJ1Y3R1cmVkPXN0cnVjdHVyZWQpClRFX0NPTFVNTlMgPSBbJ0N1aXNpbmUnLCAnUmVzdGF1cmFudCBUaGVtZScsICdPcGVuaW5nIFllYXInXQoKCmRlZiBmZWF0dXJlcyhyYXcpOgogICAgcmV0dXJuIGxlZ2FjeS5mZWF0dXJlcyhyYXcpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgpkZWYgc3RydWN0dXJlZF9mcmFtZShyYXcsIG1hcHM9Tm9uZSk6CiAgICBYID0gZmVhdHVyZXMocmF3KQogICAgaWYgbWFwcyBpcyBOb25lOgogICAgICAgIG1hcHMgPSB7YzogWFtjXS52YWx1ZV9jb3VudHMoKS50b19kaWN0KCkgZm9yIGMgaW4gWydDaXR5JywgJ0N1aXNpbmUnXX0KICAgIHNvY2lhbCA9IFhbWydGYWNlYm9vayBQb3B1bGFyaXR5IFF1b3RpZW50JywgJ0luc3RhZ3JhbSBQb3B1bGFyaXR5IFF1b3RpZW50J11dLm1lYW4oYXhpcz0xKQogICAgZGF0ZSA9IHBkLnRvX2RhdGV0aW1lKHJhd1snT3BlbmluZyBEYXkgb2YgUmVzdGF1cmFudCddLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSksIGZvcm1hdD0nJWQtJW0tJVknLCBlcnJvcnM9J2NvZXJjZScpCiAgICBhZ2UgPSAoKHBkLlRpbWVzdGFtcCgnMjAyNi0wMS0wMScpIC0gZGF0ZSkuZHQuZGF5cyAvIDM2NS4yNSkuY2xpcChsb3dlcj0wKQogICAgWFsnVjggU29jaWFsIG1lYW4nXSA9IHNvY2lhbAogICAgWFsnVjggQWdlIHllYXJzJ10gPSBhZ2UKICAgIFhbJ1Y4IFNvY2lhbCB0aWVyJ10gPSBzb2NpYWwgKiBYWydSZXN0dXJhbnQgVGllciddCiAgICBYWydWOCBTb2NpYWwgaHlnaWVuZSddID0gc29jaWFsICogWFsnSHlnaWVuZSBSYXRpbmcnXQogICAgWFsnVjggQWdlIHNvY2lhbCddID0gYWdlICogc29jaWFsCiAgICBYWydWOCBUaWVyIGNpdHkgdGllciddID0gWFsnUmVzdHVyYW50IFRpZXInXSAqIFhbJ1Jlc3RhdXJhbnQgQ2l0eSBUaWVyJ10KICAgIGZvciBjIGluIFsnQ2l0eScsICdDdWlzaW5lJ106CiAgICAgICAgWFtmJ1Y4IExvZyB7Y30gY291bnQnXSA9IG5wLmxvZzFwKFhbY10ubWFwKG1hcHNbY10pLmZpbGxuYSgwKS5hc3R5cGUoZmxvYXQpKQogICAgcmV0dXJuIFgsIG1hcHMKCgpkZWYgY2F0ZWdvcmljYWxfY29sdW1ucyhYKToKICAgIHJldHVybiBYLnNlbGVjdF9kdHlwZXMoaW5jbHVkZT1bJ29iamVjdCcsICdzdHJpbmcnLCAnY2F0ZWdvcnknXSkuY29sdW1ucy50b2xpc3QoKQoKCmRlZiBuYXRpdmVfZnJhbWUoWCwgc3RhdGU9Tm9uZSk6CiAgICAiIiJLbm93biByYXJlIGxldmVscyBwb29sIHRvZ2V0aGVyOyB0cnVseSB1bnNlZW4gbGV2ZWxzIGJlY29tZSBtaXNzaW5nLiIiIgogICAgcmVzdWx0ID0gWC5jb3B5KCkKICAgIGlmIHN0YXRlIGlzIE5vbmU6CiAgICAgICAgc3RhdGUgPSB7fQogICAgICAgIGZvciBjIGluIGNhdGVnb3JpY2FsX2NvbHVtbnMoWCk6CiAgICAgICAgICAgIGNvdW50cyA9IFhbY10udmFsdWVfY291bnRzKCkudG9fZGljdCgpCiAgICAgICAgICAgIHJhbmtlZCA9IHNvcnRlZChjb3VudHMsIGtleT1sYW1iZGEgdmFsdWU6ICgtY291bnRzW3ZhbHVlXSwgc3RyKHZhbHVlKSkpCiAgICAgICAgICAgIGtlcHQgPSBbdiBmb3IgdiBpbiByYW5rZWQgaWYgY291bnRzW3ZdID49IDEwXVs6MjUzXQogICAgICAgICAgICAjIEludGVnZXIgY29kZXMgYXZvaWQgY29sbGlzaW9ucyB3aXRoIGxpdGVyYWwgZGF0YSB2YWx1ZXMuCiAgICAgICAgICAgIG1hcHBpbmcgPSB7djogKGtlcHQuaW5kZXgodikgaWYgdiBpbiBrZXB0IGVsc2UgMjUzKSBmb3IgdiBpbiByYW5rZWR9CiAgICAgICAgICAgIHN0YXRlW2NdID0gbWFwcGluZwogICAgZm9yIGMsIG1hcHBpbmcgaW4gc3RhdGUuaXRlbXMoKToKICAgICAgICBjb2RlcyA9IFhbY10ubWFwKG1hcHBpbmcpLmFzdHlwZShmbG9hdCkKICAgICAgICByZXN1bHRbY10gPSBwZC5DYXRlZ29yaWNhbChjb2RlcywgY2F0ZWdvcmllcz1zb3J0ZWQoc2V0KG1hcHBpbmcudmFsdWVzKCkpKSkKICAgICAgICBhc3NlcnQgbGVuKHJlc3VsdFtjXS5jYXQuY2F0ZWdvcmllcykgPD0gMjU0CiAgICBmb3IgYyBpbiByZXN1bHQuY29sdW1ucy5kaWZmZXJlbmNlKGxpc3Qoc3RhdGUpKToKICAgICAgICByZXN1bHRbY10gPSBwZC50b19udW1lcmljKHJlc3VsdFtjXSwgZXJyb3JzPSdyYWlzZScpLmFzdHlwZShmbG9hdCkKICAgIHJldHVybiByZXN1bHQsIHN0YXRlCgoKZGVmIHRhcmdldF9mcmFtZShYKToKICAgIHJlc3VsdCA9IFhbVEVfQ09MVU1OU10uY29weSgpCiAgICByZXN1bHRbJ09wZW5pbmcgWWVhciddID0gcmVzdWx0WydPcGVuaW5nIFllYXInXS5tYXAobGFtYmRhIHY6ICdNaXNzaW5nJyBpZiBwZC5pc25hKHYpIGVsc2UgZid7djpnfScpCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIGZpdF9zaW5nbGUocmF3LCB5LCBzcGVjKToKICAgIFggPSBmZWF0dXJlcyhyYXcpCiAgICB0YXJnZXQgPSBucC5hc2FycmF5KHksIGZsb2F0KSAvIDFlNgogICAgdHJhbnNmb3JtZWQgPSBucC5zcXJ0KHRhcmdldCkgaWYgc3BlY1sndGFyZ2V0J10gPT0gJ3NxcnQnIGVsc2UgdGFyZ2V0CiAgICBmaXR0ZWQgPSB7J3NwZWMnOiBzcGVjLmNvcHkoKSwgJ2ZyZXF1ZW5jeV9tYXBzJzogTm9uZSwgJ2VuY29kZXInOiBOb25lLCAnbmF0aXZlX3N0YXRlJzogTm9uZX0KICAgIGlmIHNwZWNbJ2ZhbWlseSddID09ICdyaWRnZSc6CiAgICAgICAgaWYgc3BlY1snc3RydWN0dXJlZCddOgogICAgICAgICAgICBYLCBmaXR0ZWRbJ2ZyZXF1ZW5jeV9tYXBzJ10gPSBzdHJ1Y3R1cmVkX2ZyYW1lKHJhdykKICAgICAgICBjYXRzID0gY2F0ZWdvcmljYWxfY29sdW1ucyhYKQogICAgICAgIG51bXMgPSBbYyBmb3IgYyBpbiBYIGlmIGMgbm90IGluIGNhdHNdCiAgICAgICAgcHJlID0gQ29sdW1uVHJhbnNmb3JtZXIoWwogICAgICAgICAgICAoJ251bWVyaWMnLCBtYWtlX3BpcGVsaW5lKFNpbXBsZUltcHV0ZXIoc3RyYXRlZ3k9J21lZGlhbicsIGFkZF9pbmRpY2F0b3I9VHJ1ZSwga2VlcF9lbXB0eV9mZWF0dXJlcz1UcnVlKSwgU3RhbmRhcmRTY2FsZXIoKSksIG51bXMpLAogICAgICAgICAgICAoJ2NhdGVnb3J5JywgT25lSG90RW5jb2RlcihoYW5kbGVfdW5rbm93bj0naWdub3JlJywgbWluX2ZyZXF1ZW5jeT0xMCwgc3BhcnNlX291dHB1dD1GYWxzZSksIGNhdHMpXSkKICAgICAgICBtb2RlbCA9IG1ha2VfcGlwZWxpbmUocHJlLCBSaWRnZShhbHBoYT1zcGVjWydhbHBoYSddLCBmaXRfaW50ZXJjZXB0PVRydWUsIHNvbHZlcj0nY2hvbGVza3knKSkKICAgIGVsc2U6CiAgICAgICAgaWYgc3BlY1snZmFtaWx5J10gPT0gJ2VuY29kZWQnOgogICAgICAgICAgICBlbmMgPSBUYXJnZXRFbmNvZGVyKHRhcmdldF90eXBlPSdjb250aW51b3VzJywgc21vb3RoPXNwZWNbJ3Ntb290aCddLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGN2PUtGb2xkKDUsIHNodWZmbGU9VHJ1ZSwgcmFuZG9tX3N0YXRlPTgzMSkpCiAgICAgICAgICAgIHZhbHVlcyA9IGVuYy5maXRfdHJhbnNmb3JtKHRhcmdldF9mcmFtZShYKSwgdGFyZ2V0KQogICAgICAgICAgICBmaXR0ZWRbJ2VuY29kZXInXSA9IGVuYwogICAgICAgICAgICBmb3IgaiwgYyBpbiBlbnVtZXJhdGUoVEVfQ09MVU1OUyk6CiAgICAgICAgICAgICAgICBYW2YnVjggVEUge2N9J10gPSB2YWx1ZXNbOiwgal0KICAgICAgICBYLCBmaXR0ZWRbJ25hdGl2ZV9zdGF0ZSddID0gbmF0aXZlX2ZyYW1lKFgpCiAgICAgICAgbW9kZWwgPSBIaXN0R3JhZGllbnRCb29zdGluZ1JlZ3Jlc3Nvcihsb3NzPSdzcXVhcmVkX2Vycm9yJywgbGVhcm5pbmdfcmF0ZT0uMDMsCiAgICAgICAgICAgIG1heF9pdGVyPTYwMCwgbWF4X2xlYWZfbm9kZXM9c3BlY1snbGVhdmVzJ10sIG1pbl9zYW1wbGVzX2xlYWY9NTAsCiAgICAgICAgICAgIGwyX3JlZ3VsYXJpemF0aW9uPTIwLCBtYXhfYmlucz0yNTUsIGVhcmx5X3N0b3BwaW5nPUZhbHNlLAogICAgICAgICAgICBjYXRlZ29yaWNhbF9mZWF0dXJlcz0nZnJvbV9kdHlwZScsIHJhbmRvbV9zdGF0ZT1TRUVEKQogICAgbW9kZWwuZml0KFgsIHRyYW5zZm9ybWVkKQogICAgZml0dGVkWydtb2RlbCddID0gbW9kZWwKICAgIGZpdHRlZFsnY29sdW1ucyddID0gWC5jb2x1bW5zLnRvbGlzdCgpCiAgICByZXR1cm4gZml0dGVkCgoKZGVmIHByZWRpY3Rfc2luZ2xlKGZpdHRlZCwgcmF3KToKICAgIHNwZWMgPSBmaXR0ZWRbJ3NwZWMnXQogICAgWCA9IGZlYXR1cmVzKHJhdykKICAgIGlmIGZpdHRlZFsnZnJlcXVlbmN5X21hcHMnXSBpcyBub3QgTm9uZToKICAgICAgICBYLCBfID0gc3RydWN0dXJlZF9mcmFtZShyYXcsIGZpdHRlZFsnZnJlcXVlbmN5X21hcHMnXSkKICAgIGlmIGZpdHRlZFsnZW5jb2RlciddIGlzIG5vdCBOb25lOgogICAgICAgIHZhbHVlcyA9IGZpdHRlZFsnZW5jb2RlciddLnRyYW5zZm9ybSh0YXJnZXRfZnJhbWUoWCkpCiAgICAgICAgZm9yIGosIGMgaW4gZW51bWVyYXRlKFRFX0NPTFVNTlMpOgogICAgICAgICAgICBYW2YnVjggVEUge2N9J10gPSB2YWx1ZXNbOiwgal0KICAgIGlmIGZpdHRlZFsnbmF0aXZlX3N0YXRlJ10gaXMgbm90IE5vbmU6CiAgICAgICAgWCwgXyA9IG5hdGl2ZV9mcmFtZShYLCBmaXR0ZWRbJ25hdGl2ZV9zdGF0ZSddKQogICAgcCA9IG5wLm1heGltdW0oZml0dGVkWydtb2RlbCddLnByZWRpY3QoWFtmaXR0ZWRbJ2NvbHVtbnMnXV0pLCAwKQogICAgaWYgc3BlY1sndGFyZ2V0J10gPT0gJ3NxcnQnOgogICAgICAgIHAgPSBwICoqIDIKICAgIHAgPSBucC5hc2FycmF5KHAsIGZsb2F0KSAqIDFlNgogICAgYXNzZXJ0IHAuc2hhcGUgPT0gKGxlbihyYXcpLCkgYW5kIG5wLmlzZmluaXRlKHApLmFsbCgpCiAgICByZXR1cm4gcAoKCmRlZiBmaXRfcHJlZGljdChyYXcsIHksIHZhbGlkLCBzcGVjLCBtb2RlPSdwbGFpbicsIHByb2dyZXNzPU5vbmUpOgogICAgZml0dGVkID0gZml0X21vZGVsKHJhdywgeSwgc3BlYywgbW9kZSwgcHJvZ3Jlc3MpCiAgICByZXR1cm4gcHJlZGljdF9tb2RlbChmaXR0ZWQsIHZhbGlkKQoKCmRlZiBmaXRfbW9kZWwocmF3LCB5LCBzcGVjLCBtb2RlPSdwbGFpbicsIHByb2dyZXNzPU5vbmUpOgogICAgYXNzZXJ0IG1vZGUgaW4gWydwbGFpbicsICdiYWc1JywgJ21peDUwJ10KICAgIGZpdHRlZCA9IHsnbW9kZSc6IG1vZGUsICdmdWxsJzogTm9uZSwgJ2JhZ3MnOiBbXX0KICAgIGlmIG1vZGUgaW4gWydwbGFpbicsICdtaXg1MCddOgogICAgICAgIGZpdHRlZFsnZnVsbCddID0gZml0X3NpbmdsZShyYXcsIHksIHNwZWMpCiAgICBpZiBtb2RlICE9ICdwbGFpbic6CiAgICAgICAgZm9yIGksIChhLCBfKSBpbiBlbnVtZXJhdGUoS0ZvbGQoNSwgc2h1ZmZsZT1UcnVlLCByYW5kb21fc3RhdGU9ODMxKS5zcGxpdChyYXcpKToKICAgICAgICAgICAgZml0dGVkWydiYWdzJ10uYXBwZW5kKGZpdF9zaW5nbGUocmF3Lmlsb2NbYV0sIG5wLmFzYXJyYXkoeSlbYV0sIHNwZWMpKQogICAgICAgICAgICBpZiBwcm9ncmVzczoKICAgICAgICAgICAgICAgIHByb2dyZXNzKGYnYmFnIHtpICsgMX0vNSBjb21wbGV0ZScpCiAgICByZXR1cm4gZml0dGVkCgoKZGVmIHByZWRpY3RfbW9kZWwoZml0dGVkLCByYXcpOgogICAgaWYgZml0dGVkWydtb2RlJ10gPT0gJ3BsYWluJzoKICAgICAgICByZXR1cm4gcHJlZGljdF9zaW5nbGUoZml0dGVkWydmdWxsJ10sIHJhdykKICAgIHAgPSBucC5tZWFuKFtwcmVkaWN0X3NpbmdsZShtLCByYXcpIGZvciBtIGluIGZpdHRlZFsnYmFncyddXSwgYXhpcz0wKQogICAgcmV0dXJuIHAgaWYgZml0dGVkWydtb2RlJ10gPT0gJ2JhZzUnIGVsc2UgLjUgKiAocCArIHByZWRpY3Rfc2luZ2xlKGZpdHRlZFsnZnVsbCddLCByYXcpKQo=","engine.py":"IiIiSGFzaC1jaGVja2VkIGNvbXB1dGF0aW9uLCBkYXRhIGNvbnRyYWN0cyBhbmQgZnJvemVuIFY3IHJlZmVyZW5jZXMuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHNrbGVhcm4sIHNjaXB5LCBjYXRib29zdAppbXBvcnQgYmFzZWxpbmUKaW1wb3J0IGxlZ2FjeV9tb2RlbHMgYXMgbGVnYWN5CmltcG9ydCBtb2RlbGluZyBhcyBtb2RlbHMKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50ClNURU0gPSAnc3VibWlzc2lvbicKQ09ORklSTSA9IFs0ODIxMSwgNDgyMTIsIDQ4MjEzLCAnc3RyYXQ6Mjc0MjEnLCAnY2l0eSddClJBTkRPTV9DT05GSVJNID0gQ09ORklSTVs6M10KRVhQRUNURURfUlVOVElNRSA9IHsnbnVtcHknOiAnMi41LjQnLCAncGFuZGFzJzogJzMuMC42JywgJ3NrbGVhcm4nOiAnMS45LjEnLCAnc2NpcHknOiAnMS4xOC4xJywgJ2NhdGJvb3N0JzogJzEuMi4xMCd9CgoKZGVmIHNoYShwYXRoKToKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihQYXRoKHBhdGgpLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgZGlnZXN0KHZhbHVlKToKICAgIHJldHVybiBoYXNobGliLnNoYTI1Nihqc29uLmR1bXBzKHZhbHVlLCBzb3J0X2tleXM9VHJ1ZSkuZW5jb2RlKCkpLmhleGRpZ2VzdCgpCgoKZGVmIHNhdmUocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRlbXBvcmFyeSA9IHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAnLnBhcnRpYWwnKQogICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSwgaW5kZW50PTIsIGFsbG93X25hbj1GYWxzZSkgKyAnXG4nKQogICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKCgpkZWYgbG9nKHZhbHVlKToKICAgIHByaW50KHZhbHVlIGlmIGlzaW5zdGFuY2UodmFsdWUsIHN0cikgZWxzZSBqc29uLmR1bXBzKHZhbHVlKSwgZmx1c2g9VHJ1ZSkKCgpkZWYgc2FmZSh2YWx1ZSk6CiAgICByZXR1cm4gc3RyKHZhbHVlKS5yZXBsYWNlKCc6JywgJ18nKQoKCmRlZiBybXNlKHksIHApOgogICAgcmV0dXJuIGZsb2F0KG5wLnNxcnQobnAubWVhbigobnAuYXNhcnJheSh5KSAtIG5wLmFzYXJyYXkocCkpICoqIDIpKSkKCgpjbGFzcyBCdWRnZXRFeGNlZWRlZChSdW50aW1lRXJyb3IpOgogICAgcGFzcwoKCmNsYXNzIENhY2hlOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG91dCwgYnVkZ2V0X2hvdXJzPTE4KToKICAgICAgICBzZWxmLm91dCA9IFBhdGgob3V0KQogICAgICAgIHNlbGYucm9vdCA9IHNlbGYub3V0IC8gJ2NhY2hlJwogICAgICAgIHNlbGYucm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgc2VsZi5idWRnZXQgPSBzZWxmLm91dCAvICdjb21wdXRlX2J1ZGdldC5qc29uJwogICAgICAgIHNlbGYuaG91cnMgPSBidWRnZXRfaG91cnMKICAgICAgICBzZWxmLnN0YWdlID0gJ2F1ZGl0JwogICAgICAgIHNlbGYucmVhZHMgPSAwCgogICAgZGVmIHN0YXRlKHNlbGYpOgogICAgICAgIHJldHVybiBqc29uLmxvYWRzKHNlbGYuYnVkZ2V0LnJlYWRfdGV4dCgpKSBpZiBzZWxmLmJ1ZGdldC5leGlzdHMoKSBlbHNlIHsnc2Vjb25kcyc6IDAuLCAnYnlfc3RhZ2UnOiB7fX0KCiAgICBkZWYgZ2V0KHNlbGYsIG5hbWUsIGNvbnRleHQsIGNvbXB1dGUpOgogICAgICAgIHBhdGggPSBzZWxmLnJvb3QgLyAoc2FmZShuYW1lKSArICcubnB6JykKICAgICAgICBtZXRhID0gcGF0aC53aXRoX3N1ZmZpeCgnLmpzb24nKQogICAgICAgIGlmIHBhdGguZXhpc3RzKCkgYW5kIG1ldGEuZXhpc3RzKCk6CiAgICAgICAgICAgIG0gPSBqc29uLmxvYWRzKG1ldGEucmVhZF90ZXh0KCkpCiAgICAgICAgICAgIGlmIG1bJ2NvbnRleHQnXSAhPSBjb250ZXh0OgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ1N0YWxlIGNoZWNrcG9pbnQ6IHtuYW1lfScpCiAgICAgICAgICAgIGlmIG1bJ3NoYTI1NiddICE9IHNoYShwYXRoKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZidDb3JydXB0IGNoZWNrcG9pbnQ6IHtuYW1lfScpCiAgICAgICAgICAgIHAgPSBucC5sb2FkKHBhdGgsIGFsbG93X3BpY2tsZT1GYWxzZSlbJ3ByZWRpY3Rpb24nXS5jb3B5KCkKICAgICAgICAgICAgaWYgbm90IG5wLmlzZmluaXRlKHApLmFsbCgpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ0ludmFsaWQgY2hlY2twb2ludCB2YWx1ZXM6IHtuYW1lfScpCiAgICAgICAgICAgIHNlbGYucmVhZHMgKz0gMQogICAgICAgICAgICByZXR1cm4gcAogICAgICAgIHN0YXRlID0gc2VsZi5zdGF0ZSgpCiAgICAgICAgcmVzZXJ2ZWQgPSBtaW4oMy4sIHNlbGYuaG91cnMgLyA2KSAqIDM2MDAKICAgICAgICBsaW1pdCA9IHNlbGYuaG91cnMgKiAzNjAwIC0gKDAgaWYgc2VsZi5zdGFnZSA9PSAnZXhwb3J0JyBlbHNlIHJlc2VydmVkKQogICAgICAgIHN0YWdlX2xpbWl0ID0geydzY3JlZW4nOiA1LCAncmVmaW5lJzogMiwgJ2NvbmZpcm0nOiA3fS5nZXQoc2VsZi5zdGFnZSwgc2VsZi5ob3VycykgKiAzNjAwCiAgICAgICAgaWYgc3RhdGVbJ3NlY29uZHMnXSA+PSBsaW1pdCBvciBzdGF0ZVsnYnlfc3RhZ2UnXS5nZXQoc2VsZi5zdGFnZSwgMCkgPj0gc3RhZ2VfbGltaXQ6CiAgICAgICAgICAgIHJhaXNlIEJ1ZGdldEV4Y2VlZGVkKGYnQ29tcHV0ZSBidWRnZXQgcmVhY2hlZCBiZWZvcmUge25hbWV9JykKICAgICAgICBzdGFydGVkID0gdGltZS5tb25vdG9uaWMoKQogICAgICAgIHRyeToKICAgICAgICAgICAgcCA9IG5wLmFzYXJyYXkoY29tcHV0ZSgpLCBmbG9hdCkKICAgICAgICAgICAgYXNzZXJ0IG5wLmlzZmluaXRlKHApLmFsbCgpCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgc3RhdGUgPSBzZWxmLnN0YXRlKCkKICAgICAgICAgICAgc2Vjb25kcyA9IHRpbWUubW9ub3RvbmljKCkgLSBzdGFydGVkCiAgICAgICAgICAgIHN0YXRlWydzZWNvbmRzJ10gKz0gc2Vjb25kcwogICAgICAgICAgICBzdGF0ZVsnYnlfc3RhZ2UnXVtzZWxmLnN0YWdlXSA9IHN0YXRlWydieV9zdGFnZSddLmdldChzZWxmLnN0YWdlLCAwKSArIHNlY29uZHMKICAgICAgICAgICAgc3RhdGVbJ21heGltdW1faG91cnMnXSA9IHNlbGYuaG91cnMKICAgICAgICAgICAgc3RhdGVbJ21lYXN1cmVtZW50J10gPSAnQWdncmVnYXRlIGVsYXBzZWQgdGltZSBvZiBjaGVja3BvaW50IGNvbXB1dGF0aW9uczsgZnJlc2ggbm90ZWJvb2sgdmVyaWZpY2F0aW9uIGlzIHJlY29yZGVkIHNlcGFyYXRlbHkuJwogICAgICAgICAgICBzYXZlKHNlbGYuYnVkZ2V0LCBzdGF0ZSkKICAgICAgICB0ZW1wb3JhcnkgPSBwYXRoLndpdGhfc3VmZml4KCcucGFydGlhbC5ucHonKQogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQodGVtcG9yYXJ5LCBwcmVkaWN0aW9uPXApCiAgICAgICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKICAgICAgICBzYXZlKG1ldGEsIHsnY29udGV4dCc6IGNvbnRleHQsICdzaGEyNTYnOiBzaGEocGF0aCl9KQogICAgICAgIHJldHVybiBwCgoKY2xhc3MgRW5naW5lOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRhdGFfZGlyPU5vbmUsIG91dF9kaXI9Tm9uZSwgYnVkZ2V0X2hvdXJzPTE4KToKICAgICAgICBzZWxmLmRhdGFfZGlyID0gUGF0aChkYXRhX2RpciBvciBST09ULnBhcmVudCkucmVzb2x2ZSgpCiAgICAgICAgc2VsZi5vdXQgPSBQYXRoKG91dF9kaXIgb3IgUk9PVCAvICdyZXN1bHRzJykucmVzb2x2ZSgpCiAgICAgICAgc2VsZi5vdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHNlbGYudHJhaW4sIHNlbGYudGVzdCwgc2VsZi55ID0gbGVnYWN5LmxvYWRfZGF0YShzZWxmLmRhdGFfZGlyKQogICAgICAgIHNlbGYuaW5wdXRzID0ge246IHNoYShzZWxmLmRhdGFfZGlyIC8gbikgZm9yIG4gaW4gbGVnYWN5LlJFUVVJUkVEfQogICAgICAgIHNlbGYubWFuaWZlc3QgPSBqc29uLmxvYWRzKChST09UIC8gJ3JlZmVyZW5jZV9tYW5pZmVzdC5qc29uJykucmVhZF90ZXh0KCkpCiAgICAgICAgYXNzZXJ0IHNlbGYuaW5wdXRzID09IHNlbGYubWFuaWZlc3RbJ2lucHV0cyddLCAnT2ZmaWNpYWwgaW5wdXRzIGRpZmZlciBmcm9tIG1lYXN1cmVkIGRhdGEuJwogICAgICAgIHNlbGYuY29udHJhY3QgPSBqc29uLmxvYWRzKChST09UIC8gJ3Y1X2NvbnRyYWN0Lmpzb24nKS5yZWFkX3RleHQoKSkKICAgICAgICBzZWxmLnNvdXJjZXMgPSB7bjogc2hhKFJPT1QgLyBuKSBmb3IgbiBpbiBbJ2VuZ2luZS5weScsICdtb2RlbGluZy5weScsICdiYXNlbGluZS5weScsCiAgICAgICAgICAgICdsZWdhY3lfcmVmZXJlbmNlLnB5JywgJ2xlZ2FjeV9tb2RlbHMucHknLCAndjVfY29udHJhY3QuanNvbicsICdyZWZlcmVuY2VfbWFuaWZlc3QuanNvbiddfQogICAgICAgIHNlbGYucnVudGltZSA9IHsnbnVtcHknOiBucC5fX3ZlcnNpb25fXywgJ3BhbmRhcyc6IHBkLl9fdmVyc2lvbl9fLCAnc2tsZWFybic6IHNrbGVhcm4uX192ZXJzaW9uX18sCiAgICAgICAgICAgICAgICAgICAgICAgICdzY2lweSc6IHNjaXB5Ll9fdmVyc2lvbl9fLCAnY2F0Ym9vc3QnOiBjYXRib29zdC5fX3ZlcnNpb25fX30KICAgICAgICBhc3NlcnQgc2VsZi5ydW50aW1lID09IEVYUEVDVEVEX1JVTlRJTUUsIGYnSW5zdGFsbCBwaW5uZWQgd29ya2luZyB2ZXJzaW9uczoge0VYUEVDVEVEX1JVTlRJTUV9JwogICAgICAgIHNlbGYuY2FjaGUgPSBDYWNoZShzZWxmLm91dCwgYnVkZ2V0X2hvdXJzKQoKICAgIGRlZiBzcGxpdHMoc2VsZiwgc2NoZW1lKToKICAgICAgICByZXR1cm4gbGVnYWN5LnNwbGl0X2luZGljZXMoc2VsZi50cmFpbiwgc2NoZW1lKQoKICAgIGRlZiBjb250ZXh0KHNlbGYsIG5hbWUsIGEsIGIsIGV4dHJhPU5vbmUpOgogICAgICAgIHJldHVybiB7J25hbWUnOiBuYW1lLCAnaW5wdXRzJzogc2VsZi5pbnB1dHMsICdzb3VyY2VzJzogc2VsZi5zb3VyY2VzLCAncnVudGltZSc6IHNlbGYucnVudGltZSwKICAgICAgICAgICAgICAgICd0cmFpbl9pZHMnOiBkaWdlc3Qoc2VsZi50cmFpbi5pbG9jW2FdWydSZWdpc3RyYXRpb24gTnVtYmVyJ10udG9saXN0KCkpLAogICAgICAgICAgICAgICAgJ3ZhbGlkX2lkcyc6IGRpZ2VzdChzZWxmLnRyYWluLmlsb2NbYl1bJ1JlZ2lzdHJhdGlvbiBOdW1iZXInXS50b2xpc3QoKSksCiAgICAgICAgICAgICAgICAndHJhaW5pbmdfdGFyZ2V0cyc6IGhhc2hsaWIuc2hhMjU2KHNlbGYueVthXS50b2J5dGVzKCkpLmhleGRpZ2VzdCgpLCAnZXh0cmEnOiBleHRyYX0KCiAgICBkZWYgYmFzZWxpbmUoc2VsZiwgc2NoZW1lKToKICAgICAgICBmb2xkcyA9IHNlbGYuc3BsaXRzKHNjaGVtZSkKICAgICAgICBpZiBzdHIoc2NoZW1lKSBpbiBzZWxmLm1hbmlmZXN0WydzbmFwc2hvdHMnXToKICAgICAgICAgICAgbSA9IHNlbGYubWFuaWZlc3RbJ3NuYXBzaG90cyddW3N0cihzY2hlbWUpXQogICAgICAgICAgICBhc3NlcnQgbVsnaW5wdXRzJ10gPT0gc2VsZi5pbnB1dHMKICAgICAgICAgICAgYXNzZXJ0IG1bJ3RyYWluX2lkcyddID09IGRpZ2VzdChzZWxmLnRyYWluWydSZWdpc3RyYXRpb24gTnVtYmVyJ10udG9saXN0KCkpCiAgICAgICAgICAgIGFzc2VydCBtWydmb2xkX3NoYTI1NiddID09IGRpZ2VzdChbYi50b2xpc3QoKSBmb3IgXywgYiBpbiBmb2xkc10pCiAgICAgICAgICAgIHAgPSBST09UIC8gbVsnZmlsZSddCiAgICAgICAgICAgIGFzc2VydCBzaGEocCkgPT0gbVsnc2hhMjU2J10sICdDb3JydXB0IGZyb3plbiBWNyBzbmFwc2hvdCcKICAgICAgICAgICAgcmV0dXJuIG5wLmxvYWQocCwgYWxsb3dfcGlja2xlPUZhbHNlKVsncHJlZGljdGlvbiddLmNvcHkoKQogICAgICAgIHJlc3VsdCA9IG5wLnplcm9zKGxlbihzZWxmLnkpKQogICAgICAgIGZvciBmLCAoYSwgYikgaW4gZW51bWVyYXRlKGZvbGRzKToKICAgICAgICAgICAgY3R4ID0gc2VsZi5jb250ZXh0KCd2NycsIGEsIGIsIHsnc2NoZW1lJzogc3RyKHNjaGVtZSksICdmb2xkJzogZn0pCiAgICAgICAgICAgIHJlc3VsdFtiXSA9IHNlbGYuY2FjaGUuZ2V0KGYndjdfe3NhZmUoc2NoZW1lKX1fZm9sZHtmfScsIGN0eCwKICAgICAgICAgICAgICAgIGxhbWJkYTogYmFzZWxpbmUuZm9sZF9wcmVkaWN0aW9uKHNlbGYudHJhaW4uaWxvY1thXSwgc2VsZi55W2FdLCBzZWxmLnRyYWluLmlsb2NbYl0sIHNlbGYuY29udHJhY3QsIGYsCiAgICAgICAgICAgICAgICAgICAgbGFtYmRhIHM6IGxvZyhmJ1Y3IC8ge3NjaGVtZX0gLyBmb2xkIHtmICsgMX06IHtzfScpKSkKICAgICAgICAgICAgbG9nKGYnVjcgLyB7c2NoZW1lfTogZm9sZCB7ZiArIDF9LzUgY29tcGxldGUnKQogICAgICAgIHJldHVybiByZXN1bHQKCiAgICBkZWYgY29tcG9uZW50KHNlbGYsIG5hbWUsIG1vZGUsIHNjaGVtZSk6CiAgICAgICAgaWYgbW9kZSA9PSAnbWl4NTAnOgogICAgICAgICAgICAjIERlcml2aW5nIHRoaXMgYXZlcmFnZSBpcyBpZGVudGljYWwgdG8gZml0dGluZyBpdHMgdHdvIGNvbnN0aXR1ZW50IGVzdGltYXRvcnMuCiAgICAgICAgICAgIHBsYWluID0gc2VsZi5jb21wb25lbnQobmFtZSwgJ3BsYWluJywgc2NoZW1lKQogICAgICAgICAgICBiYWdnZWQgPSBzZWxmLmNvbXBvbmVudChuYW1lLCAnYmFnNScsIHNjaGVtZSkKICAgICAgICAgICAgcmV0dXJuIC41ICogKHBsYWluICsgYmFnZ2VkKQogICAgICAgIHJlc3VsdCA9IG5wLnplcm9zKGxlbihzZWxmLnkpKQogICAgICAgIHNwZWMgPSBtb2RlbHMuU1BFQ1NbbmFtZV0KICAgICAgICBmb3IgZiwgKGEsIGIpIGluIGVudW1lcmF0ZShzZWxmLnNwbGl0cyhzY2hlbWUpKToKICAgICAgICAgICAgY3R4ID0gc2VsZi5jb250ZXh0KG5hbWUsIGEsIGIsIHsnc2NoZW1lJzogc3RyKHNjaGVtZSksICdmb2xkJzogZiwgJ21vZGUnOiBtb2RlLCAnc3BlYyc6IHNwZWN9KQogICAgICAgICAgICByZXN1bHRbYl0gPSBzZWxmLmNhY2hlLmdldChmJ3tuYW1lfV97bW9kZX1fe3NhZmUoc2NoZW1lKX1fZm9sZHtmfScsIGN0eCwKICAgICAgICAgICAgICAgIGxhbWJkYTogbW9kZWxzLmZpdF9wcmVkaWN0KHNlbGYudHJhaW4uaWxvY1thXSwgc2VsZi55W2FdLCBzZWxmLnRyYWluLmlsb2NbYl0sIHNwZWMsIG1vZGUsCiAgICAgICAgICAgICAgICAgICAgbGFtYmRhIHM6IGxvZyhmJ3tuYW1lfSAvIHtzY2hlbWV9IC8gZm9sZCB7ZiArIDF9OiB7c30nKSkpCiAgICAgICAgbG9nKHsnY29tcG9uZW50JzogbmFtZSwgJ21vZGUnOiBtb2RlLCAnc2NoZW1lJzogc3RyKHNjaGVtZSksICdybXNlJzogcm1zZShzZWxmLnksIHJlc3VsdCl9KQogICAgICAgIHJldHVybiByZXN1bHQKCgpkZWYgdmFsaWRhdGVfY3N2KGZyYW1lLCB0ZXN0KToKICAgIGFzc2VydCBmcmFtZS5zaGFwZSA9PSAoNTAwLCAyKQogICAgYXNzZXJ0IGZyYW1lLmNvbHVtbnMudG9saXN0KCkgPT0gWydSZWdpc3RyYXRpb24gTnVtYmVyJywgJ0FubnVhbCBUdXJub3ZlciddCiAgICBhc3NlcnQgZnJhbWVbJ1JlZ2lzdHJhdGlvbiBOdW1iZXInXS5lcXVhbHModGVzdFsnUmVnaXN0cmF0aW9uIE51bWJlciddLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkpCiAgICBhc3NlcnQgZnJhbWVbJ1JlZ2lzdHJhdGlvbiBOdW1iZXInXS5pc191bmlxdWUKICAgIHAgPSBmcmFtZVsnQW5udWFsIFR1cm5vdmVyJ10udG9fbnVtcHkoZmxvYXQpCiAgICBhc3NlcnQgbnAuaXNmaW5pdGUocCkuYWxsKCkgYW5kIChwID49IDApLmFsbCgpCgoKZGVmIGNoZWNrX2JlaGF2aW9yKHByZWRpY3QsIHRlc3QsIHByZWRpY3Rpb24pOgogICAgcGVybSA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyg5MSkucGVybXV0YXRpb24obGVuKHRlc3QpKQogICAgYXNzZXJ0IG5wLm1heChucC5hYnMocHJlZGljdCh0ZXN0Lmlsb2NbcGVybV0pIC0gcHJlZGljdGlvbltwZXJtXSkpIDwgLjAxCiAgICBjaGFuZ2VkID0gdGVzdC5jb3B5KCkKICAgIGNoYW5nZWRbJ1JlZ2lzdHJhdGlvbiBOdW1iZXInXSA9IC03CiAgICBjaGFuZ2VkWydBbm51YWwgVHVybm92ZXInXSA9IDFlMzAKICAgIGFzc2VydCBucC5tYXgobnAuYWJzKHByZWRpY3QoY2hhbmdlZCkgLSBwcmVkaWN0aW9uKSkgPCAuMDEKICAgIGVkZ2UgPSB0ZXN0Lmlsb2NbOjNdLmNvcHkoKQogICAgZWRnZVsnQ2l0eSddID0gJ3Vuc2Vlbl9jaXR5X3Y4JwogICAgZWRnZVsnQ3Vpc2luZSddID0gcGQuTkEKICAgIGVkZ2VbJ1Jlc3RhdXJhbnQgVGhlbWUnXSA9ICd1bnNlZW5fdGhlbWVfdjgnCiAgICBlZGdlWydGYWNlYm9vayBQb3B1bGFyaXR5IFF1b3RpZW50J10gPSBucC5uYW4KICAgIGVkZ2VbJ0luc3RhZ3JhbSBQb3B1bGFyaXR5IFF1b3RpZW50J10gPSBucC5uYW4KICAgIGVkZ2VbJ1Jlc3R1cmFudCBUaWVyJ10gPSBucC5uYW4KICAgIGVkZ2VbJ09wZW5pbmcgRGF5IG9mIFJlc3RhdXJhbnQnXSA9ICdpbnZhbGlkX2RhdGUnCiAgICBlID0gcHJlZGljdChlZGdlKQogICAgYXNzZXJ0IG5wLmlzZmluaXRlKGUpLmFsbCgpIGFuZCAoZSA+PSAwKS5hbGwoKQogICAgZnJhbWUgPSBwZC5EYXRhRnJhbWUoeydSZWdpc3RyYXRpb24gTnVtYmVyJzogdGVzdFsnUmVnaXN0cmF0aW9uIE51bWJlciddLnRvX251bXB5KCksICdBbm51YWwgVHVybm92ZXInOiBwcmVkaWN0aW9ufSkKICAgIHZhbGlkYXRlX2NzdihmcmFtZSwgdGVzdCkKICAgIG1hbGZvcm1lZCA9IFtmcmFtZS5pbG9jWzotMV0sIHBkLmNvbmNhdChbZnJhbWUsIGZyYW1lLmlsb2NbOjFdXSwgaWdub3JlX2luZGV4PVRydWUpLAogICAgICAgIGZyYW1lLmlsb2NbOjotMV0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKSwgZnJhbWUuaWxvY1s6LCA6Oi0xXSwKICAgICAgICBmcmFtZS5yZW5hbWUoY29sdW1ucz17J0FubnVhbCBUdXJub3Zlcic6ICdUdXJub3Zlcid9KSwgZnJhbWUuYXNzaWduKGluZGV4PTApXQogICAgZm9yIGNvbCwgdmFsdWUgaW4gWygnUmVnaXN0cmF0aW9uIE51bWJlcicsICdzdWJtaXNzaW9uJyksICgnUmVnaXN0cmF0aW9uIE51bWJlcicsIHRlc3RbJ1JlZ2lzdHJhdGlvbiBOdW1iZXInXS5pbG9jWzFdKSwKICAgICAgICAgICAgICAgICAgICAgICAoJ0FubnVhbCBUdXJub3ZlcicsIG5wLm5hbiksICgnQW5udWFsIFR1cm5vdmVyJywgbnAuaW5mKSwgKCdBbm51YWwgVHVybm92ZXInLCAtbnAuaW5mKSwgKCdBbm51YWwgVHVybm92ZXInLCAtMS4pXToKICAgICAgICBiYWQgPSBmcmFtZS5jb3B5KCkKICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBzdHIpOgogICAgICAgICAgICBiYWRbY29sXSA9IGJhZFtjb2xdLmFzdHlwZShvYmplY3QpCiAgICAgICAgYmFkLmxvY1swLCBjb2xdID0gdmFsdWUKICAgICAgICBtYWxmb3JtZWQuYXBwZW5kKGJhZCkKICAgIGZvciBiYWQgaW4gbWFsZm9ybWVkOgogICAgICAgIHRyeToKICAgICAgICAgICAgdmFsaWRhdGVfY3N2KGJhZCwgdGVzdCkKICAgICAgICBleGNlcHQgKEFzc2VydGlvbkVycm9yLCBUeXBlRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKCdNYWxmb3JtZWQgQ1NWIGFjY2VwdGVkJykKICAgIHJldHVybiB7J3Jvd3MnOiA1MDAsICdjb2x1bW5zJzogZnJhbWUuY29sdW1ucy50b2xpc3QoKSwgJ29yaWdpbmFsX3Jvd19vcmRlcic6IFRydWUsICd1bmlxdWVfaWRzJzogVHJ1ZSwKICAgICAgICAgICAgJ2Zpbml0ZV9ub25uZWdhdGl2ZSc6IFRydWUsICdwZXJtdXRhdGlvbl9pbnZhcmlhbmNlJzogVHJ1ZSwgJ3RhcmdldF9pZF9pbnZhcmlhbmNlJzogVHJ1ZSwKICAgICAgICAgICAgJ21pc3NpbmdfYW5kX3Vuc2Vlbl92YWx1ZXMnOiBUcnVlLCAnbWFsZm9ybWVkX2Nhc2VzX3JlamVjdGVkJzogbGVuKG1hbGZvcm1lZCl9Cg==","workflow.py":"IiIiVGhlIGZpeGVkIFY4IGF1ZGl0LCBzY3JlZW5pbmcsIHJlZmluZW1lbnQsIGNvbmZpcm1hdGlvbiBhbmQgZXhwb3J0IHN0YWdlcy4iIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBzaHV0aWwKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgbW9kZWxpbmcgYXMgbW9kZWxzCmltcG9ydCBiYXNlbGluZQpmcm9tIGVuZ2luZSBpbXBvcnQgKFJPT1QsIFNURU0sIENPTkZJUk0sIFJBTkRPTV9DT05GSVJNLCBFbmdpbmUsIEJ1ZGdldEV4Y2VlZGVkLAogICAgICAgICAgICAgICAgICAgIHNoYSwgZGlnZXN0LCBzYXZlLCBsb2csIHNhZmUsIHJtc2UsIGNoZWNrX2JlaGF2aW9yLCB2YWxpZGF0ZV9jc3YpCgoKZGVmIGF1ZGl0KGUpOgogICAgbSA9IGUubWFuaWZlc3QKICAgIGZvciBuLCBleHBlY3RlZCBpbiBtWydmcm96ZW5fZmlsZXMnXS5pdGVtcygpOgogICAgICAgIGFzc2VydCBzaGEoUk9PVCAvIG4pID09IGV4cGVjdGVkCiAgICBmb3IgbiwgZXhwZWN0ZWQgaW4gbVsncHJlc2VydmVkX3Y3J10uaXRlbXMoKToKICAgICAgICBhc3NlcnQgc2hhKFJPT1QgLyAnc2NvcmVkX3Y3X2JhY2t1cCcgLyBuKSA9PSBleHBlY3RlZAogICAgcmVzdG9yZWQgPSAoUk9PVCAvICdsZWdhY3lfbW9kZWxzLnB5JykucmVhZF90ZXh0KCkucmVwbGFjZSgnaW1wb3J0IGxlZ2FjeV9yZWZlcmVuY2UgYXMgcmVmJywgJ2ltcG9ydCByZWZlcmVuY2UgYXMgcmVmJykKICAgIGltcG9ydCBoYXNobGliCiAgICBhc3NlcnQgaGFzaGxpYi5zaGEyNTYocmVzdG9yZWQuZW5jb2RlKCkpLmhleGRpZ2VzdCgpID09IG1bJ29yaWdpbmFsX3Y3X3NvdXJjZXMnXVsnbW9kZWxzLnB5J10KICAgIGFzc2VydCBzaGEoUk9PVCAvICdsZWdhY3lfcmVmZXJlbmNlLnB5JykgPT0gbVsnb3JpZ2luYWxfdjdfc291cmNlcyddWydyZWZlcmVuY2UucHknXQogICAgY2hhbmdlZCA9IGUudHJhaW4uY29weSgpCiAgICBjaGFuZ2VkWydBbm51YWwgVHVybm92ZXInXSA9IDFlMzAKICAgIGNoYW5nZWRbJ1JlZ2lzdHJhdGlvbiBOdW1iZXInXSA9IC05CiAgICBwZC50ZXN0aW5nLmFzc2VydF9mcmFtZV9lcXVhbChtb2RlbHMuZmVhdHVyZXMoY2hhbmdlZCksIG1vZGVscy5mZWF0dXJlcyhlLnRyYWluKSkKICAgIGFzc2VydCBsZW4oZS55KSA9PSAzNDkzIGFuZCBucC5pc2Nsb3NlKHJtc2UoWzAsIDNdLCBbMCwgMF0pLCBucC5zcXJ0KDQuNSkpCiAgICBzY29yZWQgPSBwZC5yZWFkX2NzdihST09UIC8gJ3Njb3JlZF92N19iYWNrdXAnIC8gZid7U1RFTX0uY3N2JykKICAgIHZhbGlkYXRlX2NzdihzY29yZWQsIGUudGVzdCkKICAgIGZvciBzY2hlbWUgaW4gWzQyLCA3MSwgJ3N0cmF0OjI3NDIxJywgJ2NpdHknXToKICAgICAgICBwID0gZS5iYXNlbGluZShzY2hlbWUpCiAgICAgICAgYXNzZXJ0IHAuc2hhcGUgPT0gZS55LnNoYXBlIGFuZCBucC5pc2Zpbml0ZShwKS5hbGwoKQogICAgcmVjb3JkID0geydpbnB1dHMnOiBlLmlucHV0cywgJ3NvdXJjZXMnOiBlLnNvdXJjZXMsICdydW50aW1lJzogZS5ydW50aW1lLAogICAgICAgICdweXRob24nOiBfX2ltcG9ydF9fKCdwbGF0Zm9ybScpLnB5dGhvbl92ZXJzaW9uKCksICdyb3dzJzogbGVuKGUueSksICd0ZXN0X3Jvd3MnOiBsZW4oZS50ZXN0KSwKICAgICAgICAnbGFiZWxzX2NoYW5nZWQnOiAwLCAndGFpbF9hYm92ZV84ME0nOiBpbnQoKGUueSA+IDgwZTYpLnN1bSgpKSwKICAgICAgICAnc2NvcmVkX3Y3X3ByZXNlcnZlZCc6IG1bJ3ByZXNlcnZlZF92NyddLCAnc2NvcmVkX3Y3X3BvcnRhbF9ybXNlJzogYmFzZWxpbmUuUE9SVEFMX1JNU0UsCiAgICAgICAgJ3V0Yyc6IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpLCAnZXh0ZXJuYWxfY29kZV9vcl9wcmVkaWN0aW9uc191c2VkJzogRmFsc2V9CiAgICBzYXZlKGUub3V0IC8gJ2F1ZGl0Lmpzb24nLCByZWNvcmQpCiAgICBwbGFuID0geydzcGVjcyc6IG1vZGVscy5TUEVDUywgJ3dlaWdodHMnOiBtb2RlbHMuV0VJR0hUUywgJ2RldmVsb3BtZW50JzogWzQyLCA3MV0sCiAgICAgICAgJ2NvbmZpcm1hdGlvbic6IENPTkZJUk0sICdtYXhfYmFzZV9zZXR0aW5ncyc6IDE2LCAnbWF4X3RvdGFsX3NldHRpbmdzJzogMjAsCiAgICAgICAgJ21heF9kZXZlbG9wbWVudF9jb25maWd1cmF0aW9ucyc6IDgwLCAncmVmaW5lbWVudF9tb2Rlcyc6IFsnYmFnNScsICdtaXg1MCddLAogICAgICAgICdzaG9ydGxpc3QnOiAnQmVzdCBlbGlnaWJsZSBoZ2IgYW5kIHJpZGdlIGdyb3VwOyBnYWluIHBvc2l0aXZlIG9uIGJvdGggZGV2ZWxvcG1lbnQgYXNzaWdubWVudHMnLAogICAgICAgICdxdWFsaWZpY2F0aW9uJzogJ0dhaW4gcG9zaXRpdmUgb24gYWxsIHRocmVlIHJhbmRvbSBjb25maXJtYXRpb25zOyBjaXR5IGFuZCBzdHJhdGlmaWVkIGRldGVyaW9yYXRpb24gPD0xJScsCiAgICAgICAgJ2J1ZGdldF9ob3Vycyc6IGUuY2FjaGUuaG91cnMsICdub19zZWFyY2hfZXhwYW5zaW9uX2FmdGVyX2NvbmZpcm1hdGlvbic6IFRydWV9CiAgICBwYXRoID0gZS5vdXQgLyAnZXhwZXJpbWVudF9wbGFuLmpzb24nCiAgICBpZiBwYXRoLmV4aXN0cygpOgogICAgICAgIGFzc2VydCBqc29uLmxvYWRzKHBhdGgucmVhZF90ZXh0KCkpID09IHBsYW4KICAgIHNhdmUocGF0aCwgcGxhbikKICAgIGxvZyh7J2F1ZGl0JzogJ1BBU1MnLCAnc2V0dGluZ3MnOiBsZW4obW9kZWxzLlNQRUNTKSwgJ3Y3X3BvcnRhbF9ybXNlJzogYmFzZWxpbmUuUE9SVEFMX1JNU0V9KQoKCmRlZiBjb21wYXJpc29uKGUsIG5hbWUsIG1vZGUsIGJhc2VsaW5lcyk6CiAgICBwID0ge3M6IGUuY29tcG9uZW50KG5hbWUsIG1vZGUsIHMpIGZvciBzIGluIFs0MiwgNzFdfQogICAgcm93cyA9IFtdCiAgICBmb3IgdyBpbiBtb2RlbHMuV0VJR0hUUzoKICAgICAgICByb3cgPSB7J25hbWUnOiBuYW1lLCAnbW9kZSc6IG1vZGUsICdncm91cCc6IG1vZGVscy5TUEVDU1tuYW1lXVsnZ3JvdXAnXSwKICAgICAgICAgICAgICAgJ2ZhbWlseSc6IG1vZGVscy5TUEVDU1tuYW1lXVsnZmFtaWx5J10sICd3ZWlnaHQnOiB3fQogICAgICAgIGZvciBzIGluIFs0MiwgNzFdOgogICAgICAgICAgICByb3dbZidybXNlX3tzfSddID0gcm1zZShlLnksICgxIC0gdykgKiBiYXNlbGluZXNbc10gKyB3ICogcFtzXSkKICAgICAgICAgICAgcm93W2YnZ2Fpbl97c30nXSA9IHJtc2UoZS55LCBiYXNlbGluZXNbc10pIC0gcm93W2Yncm1zZV97c30nXQogICAgICAgIHJvd1snbWVhbl9ybXNlJ10gPSBmbG9hdChucC5tZWFuKFtyb3dbJ3Jtc2VfNDInXSwgcm93WydybXNlXzcxJ11dKSkKICAgICAgICByb3dzLmFwcGVuZChyb3cpCiAgICByZXR1cm4gcm93cwoKCmRlZiByYW5rZWQodGFibGUpOgogICAgcmV0dXJuIHRhYmxlLnNvcnRfdmFsdWVzKFsnbWVhbl9ybXNlJywgJ3dlaWdodCcsICduYW1lJywgJ21vZGUnXSkKCgpkZWYgZWxpZ2libGUodGFibGUpOgogICAgcmV0dXJuIHJhbmtlZCh0YWJsZVsodGFibGUuZ2Fpbl80MiA+IDApICYgKHRhYmxlLmdhaW5fNzEgPiAwKV0pCgoKZGVmIHNjcmVlbihlKToKICAgIGIgPSB7czogZS5iYXNlbGluZShzKSBmb3IgcyBpbiBbNDIsIDcxXX0KICAgIHJvd3MgPSBbXQogICAgZm9yIG5hbWUgaW4gbW9kZWxzLlNQRUNTOgogICAgICAgIHJvd3MuZXh0ZW5kKGNvbXBhcmlzb24oZSwgbmFtZSwgJ3BsYWluJywgYikpCiAgICAgICAgcmFua2VkKHBkLkRhdGFGcmFtZShyb3dzKSkudG9fY3N2KGUub3V0IC8gJ3NjcmVlbmluZy5jc3YnLCBpbmRleD1GYWxzZSkKICAgICAgICBsb2coeydzY3JlZW5fY29tcGxldGUnOiBuYW1lLCAnc2V0dGluZ3NfZmluaXNoZWQnOiBsZW4ocm93cykgLy8gNCwgJ29mJzogMTZ9KQogICAgdGFibGUgPSByYW5rZWQocGQuRGF0YUZyYW1lKHJvd3MpKQogICAgc2hvcnQgPSBlbGlnaWJsZSh0YWJsZSkuZ3JvdXBieSgnZ3JvdXAnLCBzb3J0PUZhbHNlKS5oZWFkKDEpLnRvX2RpY3Qob3JpZW50PSdyZWNvcmRzJykKICAgIHNhdmUoZS5vdXQgLyAnc2NyZWVuaW5nX2RlY2lzaW9uLmpzb24nLCB7J2NvbXBsZXRlJzogVHJ1ZSwgJ3NldHRpbmdzJzogMTYsICdjb25maWd1cmF0aW9ucyc6IGxlbihyb3dzKSwgJ3Nob3J0bGlzdCc6IHNob3J0fSkKICAgIGxvZyh7J3NjcmVlbmluZ19zaG9ydGxpc3QnOiBzaG9ydH0pCgoKZGVmIHJlZmluZShlKToKICAgIHNlbGVjdGVkID0ganNvbi5sb2FkcygoZS5vdXQgLyAnc2NyZWVuaW5nX2RlY2lzaW9uLmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGFzc2VydCBzZWxlY3RlZFsnY29tcGxldGUnXSBhbmQgc2VsZWN0ZWRbJ3NldHRpbmdzJ10gPT0gMTYKICAgIGIgPSB7czogZS5iYXNlbGluZShzKSBmb3IgcyBpbiBbNDIsIDcxXX0KICAgIHJvd3MgPSBwZC5yZWFkX2NzdihlLm91dCAvICdzY3JlZW5pbmcuY3N2JykudG9fZGljdChvcmllbnQ9J3JlY29yZHMnKQogICAgZm9yIGNob2ljZSBpbiBzZWxlY3RlZFsnc2hvcnRsaXN0J106CiAgICAgICAgZm9yIG1vZGUgaW4gWydiYWc1JywgJ21peDUwJ106CiAgICAgICAgICAgIHJvd3MuZXh0ZW5kKGNvbXBhcmlzb24oZSwgY2hvaWNlWyduYW1lJ10sIG1vZGUsIGIpKQogICAgICAgICAgICByYW5rZWQocGQuRGF0YUZyYW1lKHJvd3MpKS50b19jc3YoZS5vdXQgLyAnZGV2ZWxvcG1lbnQuY3N2JywgaW5kZXg9RmFsc2UpCiAgICB0YWJsZSA9IHJhbmtlZChwZC5EYXRhRnJhbWUocm93cykpCiAgICB0YWJsZS50b19jc3YoZS5vdXQgLyAnZGV2ZWxvcG1lbnQuY3N2JywgaW5kZXg9RmFsc2UpCiAgICBzaG9ydCA9IGVsaWdpYmxlKHRhYmxlKS5ncm91cGJ5KCdncm91cCcsIHNvcnQ9RmFsc2UpLmhlYWQoMSkudG9fZGljdChvcmllbnQ9J3JlY29yZHMnKQogICAgbG9jayA9IHsnc2hvcnRsaXN0Jzogc2hvcnQsICdpbnB1dHMnOiBlLmlucHV0cywgJ3NvdXJjZXMnOiBlLnNvdXJjZXMsCiAgICAgICAgJ3BsYW5fc2hhMjU2Jzogc2hhKGUub3V0IC8gJ2V4cGVyaW1lbnRfcGxhbi5qc29uJyksICdkZXZlbG9wbWVudF9jb25maWd1cmF0aW9ucyc6IGxlbih0YWJsZSksCiAgICAgICAgJ2Zyb3plbl9iZWZvcmVfY29uZmlybWF0aW9uJzogVHJ1ZX0KICAgIHBhdGggPSBlLm91dCAvICdjYW5kaWRhdGVfbG9jay5qc29uJwogICAgaWYgcGF0aC5leGlzdHMoKToKICAgICAgICBhc3NlcnQganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dCgpKSA9PSBsb2NrLCAnRnJvemVuIHNob3J0bGlzdCBjaGFuZ2VkLicKICAgIHNhdmUocGF0aCwgbG9jaykKICAgIGxvZyh7J0ZST1pFTic6IHNob3J0LCAnbG9ja19zaGEyNTYnOiBzaGEocGF0aCksICdkZXZlbG9wbWVudF9jb25maWd1cmF0aW9ucyc6IGxlbih0YWJsZSl9KQoKCmRlZiB1bmNlcnRhaW50eSh5LCBiYXNlcywgcHJlZGljdGlvbnMpOgogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDgzMTkpCiAgICBnYWlucyA9IFtdCiAgICBmb3IgXyBpbiByYW5nZSg1MCk6CiAgICAgICAgaW5kZXggPSBybmcuaW50ZWdlcnMoMCwgbGVuKHkpLCAoMTAwLCBsZW4oeSkpKQogICAgICAgIG9uZSA9IFtdCiAgICAgICAgZm9yIHMgaW4gUkFORE9NX0NPTkZJUk06CiAgICAgICAgICAgIGJlID0gKHkgLSBiYXNlc1tzXSkgKiogMgogICAgICAgICAgICBwZSA9ICh5IC0gcHJlZGljdGlvbnNbc10pICoqIDIKICAgICAgICAgICAgb25lLmFwcGVuZChucC5zcXJ0KGJlW2luZGV4XS5tZWFuKGF4aXM9MSkpIC0gbnAuc3FydChwZVtpbmRleF0ubWVhbihheGlzPTEpKSkKICAgICAgICBnYWlucy5leHRlbmQobnAubWVhbihvbmUsIGF4aXM9MCkudG9saXN0KCkpCiAgICByZXR1cm4geydyZXBldGl0aW9ucyc6IDUwMDAsICdzZWVkJzogODMxOSwgJ2NpOTUnOiBucC5xdWFudGlsZShnYWlucywgWy4wMjUsIC45NzVdKS50b2xpc3QoKSwKICAgICAgICAgICAgJ3Bvc2l0aXZlX2ZyYWN0aW9uJzogZmxvYXQobnAubWVhbihucC5hc2FycmF5KGdhaW5zKSA+IDApKSwKICAgICAgICAgICAgJ2ludGVycHJldGF0aW9uJzogJ0NvbmRpdGlvbmFsIGpvaW50IHJlc3RhdXJhbnQtcm93IHJlc2FtcGxpbmcgYWNyb3NzIHRocmVlIGFzc2lnbm1lbnRzOyBleGNsdWRlcyBzZWFyY2gsIHJldHJhaW5pbmcgYW5kIGRpc3RyaWJ1dGlvbiB1bmNlcnRhaW50eS4nfQoKCmRlZiBjb25maXJtKGUpOgogICAgbG9jayA9IGpzb24ubG9hZHMoKGUub3V0IC8gJ2NhbmRpZGF0ZV9sb2NrLmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGFzc2VydCBsb2NrWydzb3VyY2VzJ10gPT0gZS5zb3VyY2VzIGFuZCBsb2NrWydpbnB1dHMnXSA9PSBlLmlucHV0cwogICAgcm93cywgc2VnbWVudHMsIGZvbGRzLCBkZWNpc2lvbnMgPSBbXSwgW10sIFtdLCBbXQogICAgYmFzZXMgPSB7fQogICAgZm9yIGNob2ljZSBpbiBsb2NrWydzaG9ydGxpc3QnXToKICAgICAgICBwcmVkaWN0aW9ucywgbWV0cmljcyA9IHt9LCB7fQogICAgICAgIGZvciBzY2hlbWUgaW4gQ09ORklSTToKICAgICAgICAgICAgaWYgc2NoZW1lIG5vdCBpbiBiYXNlczoKICAgICAgICAgICAgICAgIGJhc2VzW3NjaGVtZV0gPSBlLmJhc2VsaW5lKHNjaGVtZSkKICAgICAgICAgICAgYiA9IGJhc2VzW3NjaGVtZV0KICAgICAgICAgICAgYyA9IGUuY29tcG9uZW50KGNob2ljZVsnbmFtZSddLCBjaG9pY2VbJ21vZGUnXSwgc2NoZW1lKQogICAgICAgICAgICBwID0gKDEgLSBjaG9pY2VbJ3dlaWdodCddKSAqIGIgKyBjaG9pY2VbJ3dlaWdodCddICogYwogICAgICAgICAgICBwcmVkaWN0aW9uc1tzY2hlbWVdID0gcAogICAgICAgICAgICBvbGQsIG5ldyA9IHJtc2UoZS55LCBiKSwgcm1zZShlLnksIHApCiAgICAgICAgICAgIG1ldHJpY3Nbc3RyKHNjaGVtZSldID0geyd2N19ybXNlJzogb2xkLCAnY2FuZGlkYXRlX3Jtc2UnOiBuZXcsICdnYWluJzogb2xkIC0gbmV3LCAncmVsYXRpdmVfY2hhbmdlJzogbmV3IC8gb2xkIC0gMX0KICAgICAgICAgICAgcm93cy5hcHBlbmQoeyoqe2s6IGNob2ljZVtrXSBmb3IgayBpbiBbJ25hbWUnLCAnbW9kZScsICdncm91cCcsICd3ZWlnaHQnXX0sICdzY2hlbWUnOiBzdHIoc2NoZW1lKSwgKiptZXRyaWNzW3N0cihzY2hlbWUpXX0pCiAgICAgICAgICAgIGZvciBsYWJlbCwgbWFzayBpbiBbKCc8PTQwTScsIGUueSA8PSA0MGU2KSwgKCc0MC04ME0nLCAoZS55ID4gNDBlNikgJiAoZS55IDw9IDgwZTYpKSwgKCc+ODBNJywgZS55ID4gODBlNildOgogICAgICAgICAgICAgICAgdG90YWxfcmVkdWN0aW9uID0gZmxvYXQoKChlLnkgLSBiKSAqKiAyIC0gKGUueSAtIHApICoqIDIpLnN1bSgpKQogICAgICAgICAgICAgICAgcmVkdWN0aW9uID0gZmxvYXQoKChlLnlbbWFza10gLSBiW21hc2tdKSAqKiAyIC0gKGUueVttYXNrXSAtIHBbbWFza10pICoqIDIpLnN1bSgpKQogICAgICAgICAgICAgICAgc2VnbWVudHMuYXBwZW5kKHsnbmFtZSc6IGNob2ljZVsnbmFtZSddLCAnbW9kZSc6IGNob2ljZVsnbW9kZSddLCAnc2NoZW1lJzogc3RyKHNjaGVtZSksICdzZWdtZW50JzogbGFiZWwsCiAgICAgICAgICAgICAgICAgICAgJ3Jvd3MnOiBpbnQobWFzay5zdW0oKSksICd2N19ybXNlJzogcm1zZShlLnlbbWFza10sIGJbbWFza10pLCAnY2FuZGlkYXRlX3Jtc2UnOiBybXNlKGUueVttYXNrXSwgcFttYXNrXSksCiAgICAgICAgICAgICAgICAgICAgJ2NhbmRpZGF0ZV9zc2Vfc2hhcmUnOiBmbG9hdCgoKGUueVttYXNrXSAtIHBbbWFza10pICoqIDIpLnN1bSgpIC8gKChlLnkgLSBwKSAqKiAyKS5zdW0oKSksCiAgICAgICAgICAgICAgICAgICAgJ3NzZV9yZWR1Y3Rpb24nOiByZWR1Y3Rpb24sICdmcmFjdGlvbl9vZl90b3RhbF9yZWR1Y3Rpb24nOiByZWR1Y3Rpb24gLyB0b3RhbF9yZWR1Y3Rpb24gaWYgdG90YWxfcmVkdWN0aW9uIGVsc2UgTm9uZX0pCiAgICAgICAgICAgIGZvciBmLCAoXywgaW5kaWNlcykgaW4gZW51bWVyYXRlKGUuc3BsaXRzKHNjaGVtZSkpOgogICAgICAgICAgICAgICAgZm9sZHMuYXBwZW5kKHsnbmFtZSc6IGNob2ljZVsnbmFtZSddLCAnbW9kZSc6IGNob2ljZVsnbW9kZSddLCAnc2NoZW1lJzogc3RyKHNjaGVtZSksICdmb2xkJzogZiwKICAgICAgICAgICAgICAgICAgICAncm93cyc6IGxlbihpbmRpY2VzKSwgJ3RhaWxfcm93cyc6IGludCgoZS55W2luZGljZXNdID4gODBlNikuc3VtKCkpLAogICAgICAgICAgICAgICAgICAgICd2N19ybXNlJzogcm1zZShlLnlbaW5kaWNlc10sIGJbaW5kaWNlc10pLCAnY2FuZGlkYXRlX3Jtc2UnOiBybXNlKGUueVtpbmRpY2VzXSwgcFtpbmRpY2VzXSl9KQogICAgICAgICAgICBwZC5EYXRhRnJhbWUocm93cykudG9fY3N2KGUub3V0IC8gJ2NvbmZpcm1hdGlvbi5jc3YnLCBpbmRleD1GYWxzZSkKICAgICAgICAgICAgcGQuRGF0YUZyYW1lKHNlZ21lbnRzKS50b19jc3YoZS5vdXQgLyAnc2VnbWVudHMuY3N2JywgaW5kZXg9RmFsc2UpCiAgICAgICAgICAgIHBkLkRhdGFGcmFtZShmb2xkcykudG9fY3N2KGUub3V0IC8gJ2ZvbGRzLmNzdicsIGluZGV4PUZhbHNlKQogICAgICAgICAgICBsb2coeydjb25maXJtYXRpb24nOiBjaG9pY2VbJ25hbWUnXSwgJ3NjaGVtZSc6IHN0cihzY2hlbWUpLCAqKm1ldHJpY3Nbc3RyKHNjaGVtZSldfSkKICAgICAgICBxdWFsaWZpZWQgPSBhbGwobWV0cmljc1tzdHIocyldWydnYWluJ10gPiAwIGZvciBzIGluIFJBTkRPTV9DT05GSVJNKSBhbmQgYWxsKAogICAgICAgICAgICBtZXRyaWNzW3N0cihzKV1bJ3JlbGF0aXZlX2NoYW5nZSddIDw9IC4wMSBmb3IgcyBpbiBbJ3N0cmF0OjI3NDIxJywgJ2NpdHknXSkKICAgICAgICBkZWNpc2lvbnMuYXBwZW5kKHsqKmNob2ljZSwgJ21ldHJpY3MnOiBtZXRyaWNzLCAncXVhbGlmaWVkJzogYm9vbChxdWFsaWZpZWQpLAogICAgICAgICAgICAnbWVhbl9yYW5kb21fZ2Fpbic6IGZsb2F0KG5wLm1lYW4oW21ldHJpY3Nbc3RyKHMpXVsnZ2FpbiddIGZvciBzIGluIFJBTkRPTV9DT05GSVJNXSkpLAogICAgICAgICAgICAndW5jZXJ0YWludHknOiB1bmNlcnRhaW50eShlLnksIGJhc2VzLCBwcmVkaWN0aW9ucyl9KQogICAgdHJpYWxzID0gc29ydGVkKFtkIGZvciBkIGluIGRlY2lzaW9ucyBpZiBkWydxdWFsaWZpZWQnXV0sIGtleT1sYW1iZGEgZDogKC1kWydtZWFuX3JhbmRvbV9nYWluJ10sIGRbJ3dlaWdodCddLCBkWyduYW1lJ10pKQogICAgc2F2ZShlLm91dCAvICdkZWNpc2lvbi5qc29uJywgeydjYW5kaWRhdGVzJzogZGVjaXNpb25zLCAndHJpYWxzJzogdHJpYWxzLAogICAgICAgICdsb2NrX3NoYTI1Nic6IHNoYShlLm91dCAvICdjYW5kaWRhdGVfbG9jay5qc29uJyksICdiYXNlbGluZV9wb3J0YWxfcm1zZSc6IGJhc2VsaW5lLlBPUlRBTF9STVNFLAogICAgICAgICdkZWNpc2lvbic6ICdxdWFsaWZpZWQgbW9kZWwtZ2VuZXJhdGVkIHBvcnRhbCB0cmlhbHMnIGlmIHRyaWFscyBlbHNlICdyZXRhaW4gc2NvcmVkIFY3OyBubyBWOCBjYW5kaWRhdGUgcGFzc2VkIGFsbCBnYXRlcycsCiAgICAgICAgJ3BvcnRhbF91cGxvYWRzX2J5X2FnZW50JzogMH0pCiAgICBsb2coeydERUNJU0lPTic6ICdxdWFsaWZpZWQgcG9ydGFsIHRyaWFscycgaWYgdHJpYWxzIGVsc2UgJ3JldGFpbiBWNycsICdxdWFsaWZpZWQnOiBsZW4odHJpYWxzKX0pCgoKZGVmIF9leHBvcnQoZSk6CiAgICBkZWNpc2lvbiA9IGpzb24ubG9hZHMoKGUub3V0IC8gJ2RlY2lzaW9uLmpzb24nKS5yZWFkX3RleHQoKSkKICAgICMgQWx3YXlzIHBlcmZvcm0gc2VwYXJhdGUgZnJlc2ggVjcgcmVwcm9kdWN0aW9uLCBldmVuIHdoZW4gbm8gVjggbW9kZWwgcXVhbGlmaWVzLgogICAgbG9nKCdGcmVzaCBzY29yZWQtVjcgZnVsbCByZXByb2R1Y3Rpb24gYmVnaW5zLicpCiAgICBzdGFydGVkID0gX19pbXBvcnRfXygndGltZScpLm1vbm90b25pYygpCiAgICBiLCBmaXR0ZWRfYmFzZSA9IGJhc2VsaW5lLmZpdF9mdWxsKGUudHJhaW4sIGUudGVzdCwgZS55LCBlLmNvbnRyYWN0LCBsb2cpCiAgICBleHBlY3RlZCA9IHBkLnJlYWRfY3N2KFJPT1QgLyAnc2NvcmVkX3Y3X2JhY2t1cCcgLyBmJ3tTVEVNfS5jc3YnKVsnQW5udWFsIFR1cm5vdmVyJ10udG9fbnVtcHkoZmxvYXQpCiAgICBkaWZmZXJlbmNlID0gZmxvYXQobnAubWF4KG5wLmFicyhiIC0gZXhwZWN0ZWQpKSkKICAgIGFzc2VydCBkaWZmZXJlbmNlIDwgLjAxCiAgICBiYXNlX2NoZWNrcyA9IGNoZWNrX2JlaGF2aW9yKGxhbWJkYSByYXc6IGJhc2VsaW5lLnByZWRpY3QoZml0dGVkX2Jhc2UsIHJhdyksIGUudGVzdCwgYikKICAgIHNhdmUoZS5vdXQgLyAndjdfZnVsbF9yZXByb2R1Y3Rpb24uanNvbicsIHsnbWF4aW11bV9kaWZmZXJlbmNlX0lOUic6IGRpZmZlcmVuY2UsCiAgICAgICAgJ2ZyZXNoX2Z1bGxfZml0JzogVHJ1ZSwgJ3NlY29uZHMnOiBfX2ltcG9ydF9fKCd0aW1lJykubW9ub3RvbmljKCkgLSBzdGFydGVkLCAnY2hlY2tzJzogYmFzZV9jaGVja3N9KQogICAgY2hlY2tzID0gW10KICAgIGZvciBpLCBjaG9pY2UgaW4gZW51bWVyYXRlKGRlY2lzaW9uWyd0cmlhbHMnXSwgMSk6CiAgICAgICAgZml0dGVkID0gbW9kZWxzLmZpdF9tb2RlbChlLnRyYWluLCBlLnksIG1vZGVscy5TUEVDU1tjaG9pY2VbJ25hbWUnXV0sIGNob2ljZVsnbW9kZSddLCBsb2cpCiAgICAgICAgcCA9ICgxIC0gY2hvaWNlWyd3ZWlnaHQnXSkgKiBiICsgY2hvaWNlWyd3ZWlnaHQnXSAqIG1vZGVscy5wcmVkaWN0X21vZGVsKGZpdHRlZCwgZS50ZXN0KQogICAgICAgIHByZWRpY3QgPSBsYW1iZGEgcmF3OiAoKDEgLSBjaG9pY2VbJ3dlaWdodCddKSAqIGJhc2VsaW5lLnByZWRpY3QoZml0dGVkX2Jhc2UsIHJhdykKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKyBjaG9pY2VbJ3dlaWdodCddICogbW9kZWxzLnByZWRpY3RfbW9kZWwoZml0dGVkLCByYXcpKQogICAgICAgIGNoZWNrID0gY2hlY2tfYmVoYXZpb3IocHJlZGljdCwgZS50ZXN0LCBwKQogICAgICAgIGRpcmVjdG9yeSA9IGUub3V0LnBhcmVudCAvICdjYW5kaWRhdGVzJyAvIGYndHJpYWxfe2l9JwogICAgICAgIGRpcmVjdG9yeS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgcGF0aCA9IGRpcmVjdG9yeSAvIGYne1NURU19LmNzdicKICAgICAgICBwZC5EYXRhRnJhbWUoeydSZWdpc3RyYXRpb24gTnVtYmVyJzogZS50ZXN0WydSZWdpc3RyYXRpb24gTnVtYmVyJ10sICdBbm51YWwgVHVybm92ZXInOiBwfSkudG9fY3N2KHBhdGgsIGluZGV4PUZhbHNlKQogICAgICAgIHZhbGlkYXRlX2NzdihwZC5yZWFkX2NzdihwYXRoKSwgZS50ZXN0KQogICAgICAgIGFzc2VydCBucC5tYXgobnAuYWJzKHBkLnJlYWRfY3N2KHBhdGgpWydBbm51YWwgVHVybm92ZXInXSAtIHApKSA8IC4wMQogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQoZS5vdXQgLyBmJ2Z1bGxfZml0X3RyaWFsX3tpfS5ucHonLCBwcmVkaWN0aW9uPXAsIGJhc2VsaW5lPWIpCiAgICAgICAgY2hhbmdlID0gZmxvYXQobnAuc3FydChucC5tZWFuKChwIC0gYikgKiogMikpKQogICAgICAgIGNoZWNrLnVwZGF0ZSh0cmlhbD1pLCBjc3Y9c3RyKHBhdGgpLCBjc3Zfc2hhMjU2PXNoYShwYXRoKSwgY2hvaWNlPWNob2ljZSwKICAgICAgICAgICAgcHJlZGljdGlvbl9ybXNfZGlmZmVyZW5jZV9mcm9tX3Y3PWNoYW5nZSwgY29uZGl0aW9uYWxfcG9ydGFsX3Jtc2VfbG93ZXJfYm91bmQ9bWF4KDAsIGJhc2VsaW5lLlBPUlRBTF9STVNFIC0gY2hhbmdlKSkKICAgICAgICBjaGVja3MuYXBwZW5kKGNoZWNrKQogICAgICAgIGxvZyh7J2V4cG9ydGVkJzogc3RyKHBhdGgpLCAnY29uZGl0aW9uYWxfcG9ydGFsX3Jtc2VfbG93ZXJfYm91bmQnOiBjaGVja1snY29uZGl0aW9uYWxfcG9ydGFsX3Jtc2VfbG93ZXJfYm91bmQnXX0pCiAgICBzYXZlKGUub3V0IC8gJ3N1Ym1pc3Npb25fY2hlY2tzLmpzb24nLCBjaGVja3MpCiAgICBpZiBub3QgZGVjaXNpb25bJ3RyaWFscyddOgogICAgICAgIHJlY29tbWVuZGVkID0gZS5vdXQucGFyZW50IC8gJ3JlY29tbWVuZGVkX3N1Ym1pc3Npb24nCiAgICAgICAgcmVjb21tZW5kZWQubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBmb3IgZXh0IGluIFsnY3N2JywgJ2lweW5iJ106CiAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZShST09UIC8gJ3Njb3JlZF92N19iYWNrdXAnIC8gZid7U1RFTX0ue2V4dH0nLCByZWNvbW1lbmRlZCAvIGYne1NURU19LntleHR9JykKICAgIGxvZyh7J2V4cG9ydCc6ICdjb21wbGV0ZScsICdxdWFsaWZpZWRfZmlsZXMnOiBsZW4oY2hlY2tzKSwgJ3Y3X3JlcHJvZHVjdGlvbl9tYXhfZGlmZmVyZW5jZSc6IGRpZmZlcmVuY2V9KQoKCmRlZiBleHBvcnQoZSk6CiAgICBpbXBvcnQgdGltZQogICAgc3RhdGUgPSBlLmNhY2hlLnN0YXRlKCkKICAgIGlmIHN0YXRlWydzZWNvbmRzJ10gPj0gZS5jYWNoZS5ob3VycyAqIDM2MDA6CiAgICAgICAgcmFpc2UgQnVkZ2V0RXhjZWVkZWQoJ05vIGNvbXB1dGUgYnVkZ2V0IHJlbWFpbnMgZm9yIGZyZXNoIGV4cG9ydC4nKQogICAgc3RhcnRlZCA9IHRpbWUubW9ub3RvbmljKCkKICAgIHRyeToKICAgICAgICBfZXhwb3J0KGUpCiAgICBmaW5hbGx5OgogICAgICAgIHN0YXRlID0gZS5jYWNoZS5zdGF0ZSgpCiAgICAgICAgZWxhcHNlZCA9IHRpbWUubW9ub3RvbmljKCkgLSBzdGFydGVkCiAgICAgICAgc3RhdGVbJ3NlY29uZHMnXSArPSBlbGFwc2VkCiAgICAgICAgc3RhdGVbJ2J5X3N0YWdlJ11bJ2V4cG9ydCddID0gc3RhdGVbJ2J5X3N0YWdlJ10uZ2V0KCdleHBvcnQnLCAwKSArIGVsYXBzZWQKICAgICAgICBzdGF0ZVsnbWF4aW11bV9ob3VycyddID0gZS5jYWNoZS5ob3VycwogICAgICAgIHN0YXRlWydtZWFzdXJlbWVudCddID0gJ0FnZ3JlZ2F0ZSBjaGVja3BvaW50LWNvbXB1dGF0aW9uIGFuZCBmcmVzaC1leHBvcnQgZWxhcHNlZCBzZWNvbmRzOyBpc29sYXRlZCBub3RlYm9vayBleGVjdXRpb24gcmVjb3JkZWQgc2VwYXJhdGVseS4nCiAgICAgICAgc2F2ZShlLmNhY2hlLmJ1ZGdldCwgc3RhdGUpCgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCdzdGFnZScsIGNob2ljZXM9WydhdWRpdCcsICdzY3JlZW4nLCAncmVmaW5lJywgJ2NvbmZpcm0nLCAnZXhwb3J0JywgJ2FsbCddKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1kYXRhLWRpcicsIHR5cGU9UGF0aCwgZGVmYXVsdD1ST09ULnBhcmVudCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tb3V0LWRpcicsIHR5cGU9UGF0aCwgZGVmYXVsdD1ST09UIC8gJ3Jlc3VsdHMnKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1idWRnZXQtaG91cnMnLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTE4KQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIGFzc2VydCAwIDwgYXJncy5idWRnZXRfaG91cnMgPD0gMTgKICAgIGUgPSBFbmdpbmUoYXJncy5kYXRhX2RpciwgYXJncy5vdXRfZGlyLCBhcmdzLmJ1ZGdldF9ob3VycykKICAgIGZvciBzdGFnZSBpbiAoWydhdWRpdCcsICdzY3JlZW4nLCAncmVmaW5lJywgJ2NvbmZpcm0nLCAnZXhwb3J0J10gaWYgYXJncy5zdGFnZSA9PSAnYWxsJyBlbHNlIFthcmdzLnN0YWdlXSk6CiAgICAgICAgZS5jYWNoZS5zdGFnZSA9IHN0YWdlCiAgICAgICAgdHJ5OgogICAgICAgICAgICBnbG9iYWxzKClbc3RhZ2VdKGUpCiAgICAgICAgZXhjZXB0IEJ1ZGdldEV4Y2VlZGVkIGFzIGVycm9yOgogICAgICAgICAgICBzYXZlKGUub3V0IC8gJ2J1ZGdldF9zdG9wLmpzb24nLCB7J3N0YWdlJzogc3RhZ2UsICdyZWFzb24nOiBzdHIoZXJyb3IpLCAnY29tcGxldGVkX2NhbmRpZGF0ZXNfb25seSc6IFRydWV9KQogICAgICAgICAgICBpZiBzdGFnZSAhPSAnZXhwb3J0JzoKICAgICAgICAgICAgICAgIHNhdmUoZS5vdXQgLyAnZGVjaXNpb24uanNvbicsIHsnY2FuZGlkYXRlcyc6IFtdLCAndHJpYWxzJzogW10sICdkZWNpc2lvbic6ICdyZXRhaW4gVjc6IHJlc2VhcmNoIHN0b3BwZWQgYXQgY29tcHV0ZSBidWRnZXQnLCAncG9ydGFsX3VwbG9hZHNfYnlfYWdlbnQnOiAwfSkKICAgICAgICAgICAgICAgIGUuY2FjaGUuc3RhZ2UgPSAnZXhwb3J0JwogICAgICAgICAgICAgICAgZXhwb3J0KGUpCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICByYWlzZQo=","reference_manifest.json":"...","v5_contract.json":"ewogICJ2Ml9jb25maWdzIjogewogICAgImVuZ2luZWVyZWQiOiB7CiAgICAgICJraW5kIjogImNhdCIsCiAgICAgICJ2YXJpYW50IjogImVuZ2luZWVyZWQiLAogICAgICAiZGVwdGgiOiA2LAogICAgICAiaXRlcmF0aW9ucyI6IDI2MAogICAgfSwKICAgICJlbmdpbmVlcmVkX2Ryb3AiOiB7CiAgICAgICJraW5kIjogImNhdCIsCiAgICAgICJ2YXJpYW50IjogImVuZ2luZWVyZWRfZHJvcCIsCiAgICAgICJkZXB0aCI6IDYsCiAgICAgICJpdGVyYXRpb25zIjogMjE2CiAgICB9LAogICAgIm9yZGVyZWRfZGVwdGg1IjogewogICAgICAia2luZCI6ICJjYXQiLAogICAgICAidmFyaWFudCI6ICJiYXNlIiwKICAgICAgImRlcHRoIjogNSwKICAgICAgImJvb3N0aW5nX3R5cGUiOiAiT3JkZXJlZCIsCiAgICAgICJpdGVyYXRpb25zIjogMjE5CiAgICB9LAogICAgImh1YmVyMjBfZW5naW5lZXJlZCI6IHsKICAgICAgImtpbmQiOiAiY2F0IiwKICAgICAgInZhcmlhbnQiOiAiZW5naW5lZXJlZF9kcm9wIiwKICAgICAgImRlcHRoIjogNSwKICAgICAgImxvc3MiOiAiSHViZXI6ZGVsdGE9MjAwMDAwMDAiLAogICAgICAiaXRlcmF0aW9ucyI6IDY5OQogICAgfSwKICAgICJodWJlcjQwIjogewogICAgICAia2luZCI6ICJjYXQiLAogICAgICAidmFyaWFudCI6ICJiYXNlIiwKICAgICAgImRlcHRoIjogNSwKICAgICAgImxvc3MiOiAiSHViZXI6ZGVsdGE9NDAwMDAwMDAiLAogICAgICAiaXRlcmF0aW9ucyI6IDY2NAogICAgfSwKICAgICJjbGlwMTIwIjogewogICAgICAia2luZCI6ICJjYXQiLAogICAgICAidmFyaWFudCI6ICJiYXNlIiwKICAgICAgImRlcHRoIjogNSwKICAgICAgImNhcCI6IDEyMDAwMDAwMC4wLAogICAgICAiaXRlcmF0aW9ucyI6IDU5MQogICAgfSwKICAgICJzcGxpbmVfcmlkZ2UiOiB7CiAgICAgICJraW5kIjogInNwbGluZSIsCiAgICAgICJ2YXJpYW50IjogImJhc2UiLAogICAgICAiYWxwaGEiOiAxMDAKICAgIH0sCiAgICAic3ZyX2NvcmUiOiB7CiAgICAgICJraW5kIjogInN2ciIsCiAgICAgICJ2YXJpYW50IjogImNvcmUiLAogICAgICAiQyI6IDIwCiAgICB9LAogICAgImh1YmVyNDBfZW5naW5lZXJlZCI6IHsKICAgICAgImtpbmQiOiAiY2F0IiwKICAgICAgInZhcmlhbnQiOiAiZW5naW5lZXJlZCIsCiAgICAgICJkZXB0aCI6IDUsCiAgICAgICJsb3NzIjogIkh1YmVyOmRlbHRhPTQwMDAwMDAwIiwKICAgICAgIml0ZXJhdGlvbnMiOiA2MTkKICAgIH0sCiAgICAiaHViZXI0MF9tb25vdG9uZSI6IHsKICAgICAgImtpbmQiOiAiY2F0IiwKICAgICAgInZhcmlhbnQiOiAiYmFzZSIsCiAgICAgICJkZXB0aCI6IDUsCiAgICAgICJsb3NzIjogIkh1YmVyOmRlbHRhPTQwMDAwMDAwIiwKICAgICAgIm1vbm90b25lIjogdHJ1ZSwKICAgICAgIml0ZXJhdGlvbnMiOiA0NDcKICAgIH0KICB9LAogICJ2Ml93ZWlnaHRzIjogewogICAgImVuZ2luZWVyZWQiOiAwLjA2NDQ4OTA3MjQ5NDA4MzYyLAogICAgImVuZ2luZWVyZWRfZHJvcCI6IDAuMDk2MzE4MzQxNjMxNTMxOTcsCiAgICAib3JkZXJlZF9kZXB0aDUiOiAwLjA4NTA3MDM5MTc3Nzc4NzEyLAogICAgImh1YmVyMjBfZW5naW5lZXJlZCI6IDAuMDYyNzQ0NTI4NjEyODIwNDgsCiAgICAiaHViZXI0MCI6IDAuMDg4NTUzNjI2MDc5ODUzNzIsCiAgICAiY2xpcDEyMCI6IDAuMDc3OTQ2MzE1NTg4MTQzNjMsCiAgICAic3BsaW5lX3JpZGdlIjogMC4xNTY0ODE4NTQzMTA4NDMzOCwKICAgICJzdnJfY29yZSI6IDAuMDU4NDI4Mjk0Mzc0MjkyNzE0LAogICAgImh1YmVyNDBfZW5naW5lZXJlZCI6IDAuMDkzMjc3NDc5NjY2MDI0ODMsCiAgICAiaHViZXI0MF9tb25vdG9uZSI6IDAuMjE2NjkwMDk1NDY0NjE4NQogIH0sCiAgInYzX3BhcmFtZXRlcnMiOiB7CiAgICAidmFyaWFudCI6ICJiYXNlIiwKICAgICJkZXB0aCI6IDUsCiAgICAiaXRlcmF0aW9ucyI6IDg1MCwKICAgICJkZWx0YSI6IDQwLAogICAgInN1YnNhbXBsZSI6IDAuNywKICAgICJsMiI6IDEwCiAgfSwKICAidjNfc2VlZHMiOiBbCiAgICAxMDQyLAogICAgMjA0MiwKICAgIDMwNDIKICBdLAogICJ2ZXJzaW9uIjogInY1IiwKICAidjNfd2VpZ2h0IjogMC4yNSwKICAic2VlZHMiOiBbCiAgICAxMDI0MiwKICAgIDIwMjQyCiAgXSwKICAiY29tcG9uZW50cyI6IHsKICAgICJzcXJ0X2Q2X2NvcnJlY3QxIjogewogICAgICAic3BlYyI6IHsKICAgICAgICAicG93ZXIiOiAwLjUsCiAgICAgICAgImRlcHRoIjogNiwKICAgICAgICAiY29ycmVjdF9tZWFuIjogdHJ1ZQogICAgICB9LAogICAgICAid2VpZ2h0IjogMC4yNQogICAgfSwKICAgICJzdWJzcGFjZV9oNDAiOiB7CiAgICAgICJzcGVjIjogewogICAgICAgICJsb3NzIjogIkh1YmVyOmRlbHRhPTQwIiwKICAgICAgICAicnNtIjogMC41LAogICAgICAgICJpdGVyYXRpb25zIjogMTIwMAogICAgICB9LAogICAgICAid2VpZ2h0IjogMC4yNQogICAgfSwKICAgICJxdWFudGlsZTAuNiI6IHsKICAgICAgInNwZWMiOiB7CiAgICAgICAgImxvc3MiOiAiUXVhbnRpbGU6YWxwaGE9MC42IiwKICAgICAgICAiZGVwdGgiOiA1CiAgICAgIH0sCiAgICAgICJ3ZWlnaHQiOiAwLjI1CiAgICB9CiAgfSwKICAiZGV2ZWxvcG1lbnQiOiB7CiAgICAic2V0dGluZ3MiOiAyMiwKICAgICJzaG9ydGxpc3RlZF9zZXR0aW5ncyI6IDYsCiAgICAic3BsaXRzIjogWwogICAgICA0MiwKICAgICAgNzEKICAgIF0sCiAgICAic2VsZWN0aW9uIjogewogICAgICAibmFtZSI6ICJ0b3AzIiwKICAgICAgIm5ld193ZWlnaHQiOiAwLjc1LAogICAgICAicm1zZTQyIjogMTk4MTcyMzYuMzI3NTgxOTk4LAogICAgICAicm1zZTcxIjogMTk4NDAyNTAuMzYxNjU5MTQzLAogICAgICAibWVhbl9ybXNlIjogMTk4Mjg3NDMuMzQ0NjIwNTcsCiAgICAgICJnYWluNDIiOiAyMTc5MS4yMTMyNTkyOTgzNTQsCiAgICAgICJnYWluNzEiOiAyOTgzMy4yNDYwMDUzMTUzMzQKICAgIH0sCiAgICAic2NyZWVuaW5nX2lzX29wdGltaXN0aWMiOiB0cnVlCiAgfSwKICAiY29uZmlybWF0aW9uIjogewogICAgIm5ld19zcGxpdHMiOiBbCiAgICAgIDE4NDIxLAogICAgICAxODQyMgogICAgXSwKICAgICJwcmV2aW91c19zcGxpdHMiOiBbCiAgICAgIDIwMjYsCiAgICAgIDMwMwogICAgXSwKICAgICJjaXR5X3N0cmVzcyI6IHRydWUsCiAgICAiYWxsX3NwbGl0c19yZXVzZV9wcmV2aW91c2x5X3NlZW5fbGFiZWxzIjogdHJ1ZQogIH0KfQo=","baseline_snapshots/v7_42.npz":"...","baseline_snapshots/v7_71.npz":"...","baseline_snapshots/v7_city.npz":"...","baseline_snapshots/v7_strat_27421.npz":"...","scored_v7_backup/submission.csv":"UmVnaXN0cmF0aW9uIE51bWJlcixBbm51YWwgVHVybm92ZXIKMjAwMDEsMzA1ODg5OTAuMTUxODM3NDU3CjIwMDAyLDM5OTIwNjMwLjAzODAzNTIxCjIwMDAzLDI4NjM2MzYwLjEzNjg4MDgzCjIwMDA0LDQ0MDg2MTU0LjY2OTIxNTYyCjIwMDA1LDQ4MTE4NDg4Ljg0MTM1NDQKMjAwMDYsMzc3NDAyNDMuNzU4MTI2MTcKMjAwMDcsMzM4MDE2MDYuNDI5Mzg4MzcKMjAwMDgsMzQwMTAyNDAuMTU3OTE3NjMKMjAwMDksMjkxNjMzMzQuMjI1Mjg5MzE1CjIwMDEwLDI1NDE1NzM5Ljk3ODk4OTg0CjIwMDExLDI4Mzk2MzIwLjY0MDEwMDQyCjIwMDEyLDM0MTA1ODkzLjk2NjczMzkKMjAwMTMsMzYwNjc4NjguOTk2MDE2MzcKMjAwMTQsMjI2MzIyOTYuNzU5OTYxNDI2CjIwMDE1LDE5NzE3NzExLjkyNjExNjQyNgoyMDAxNiw0MTI4NjA4NC4xMzkxNjExODQKMjAwMTcsMzkzNjA5ODQuOTE2NjY4NwoyMDAxOCwxNTQxMjI3Ni4yNjMxNjMyMDIKMjAwMTksMzY0MzQ4ODguNzU2NzU0NTMKMjAwMjAsMjg4Mzg0NDMuNzkzMzc1MTc1CjIwMDIxLDE3NTc4NTA4LjU1ODY0MDUxCjIwMDIyLDMwODE5MjM2Ljc2NDI2NzEwMgoyMDAyMywyNTAzMjMwNS45Mzg5MTY1MzQKMjAwMjQsMjI2OTA1NzAuMjExODQ2NjYKMjAwMjUsMzkxNzQwMTkuNDU2ODI0NjMKMjAwMjYsMTYxNTA0ODcuMDMwMDI0MDU1CjIwMDI3LDM4NDAzMDQzLjc2MTk2NDgKMjAwMjgsMzY4MTA4NTYuNzAxODAzNjI1CjIwMDI5LDMxMDYxMDEyLjA1NjY4ODg2CjIwMDMwLDIxMDE2ODQ2LjQ5MTE3Nzk3NgoyMDAzMSw0MjYxNzU1My40NzcwOTg2NgoyMDAzMiwzODEwNzg0OC43MDIxODY0MDYKMjAwMzMsMjc5MjU1ODkuMjE2OTQzNjYKMjAwMzQsMzcyODY1NDEuMTE2MTEyNzEKMjAwMzUsMjE1MjE5MzcuMTU2Nzc1MDgKMjAwMzYsMzEwNTk5MjYuNjA0NTYwMjUKMjAwMzcsNDA3MDU3NzIuNzg4NjI2NTYKMjAwMzgsMzQxMTc0NTEuOTA0MzE2MzgKMjAwMzksMjExNDIzOTEuMDY1MTY2Nzg2CjIwMDQwLDMzNjI4ODYwLjQxMTgzNTczCjIwMDQxLDI3OTk1ODMyLjg3MzY0MjQ5MwoyMDA0MiwyODIyNDIwNy41NDc5ODU3NDcKMjAwNDMsMjYzMDkxNjAuMjA0MDIwOTg0CjIwMDQ0LDI2NTc3MjEwLjIwOTU4Mjc5CjIwMDQ1LDM5NjEzMzE4LjIyOTA4NTk3CjIwMDQ2LDE5MDk4NDc4LjMwNjY2MDIwNQoyMDA0NywyNjcyNTg5Mi44MTM4MzgzNzgKMjAwNDgsMzc2ODk3NjcuMjY2NDQzOTcKMjAwNDksMjQwNzIyMDcuMzU0MDIyNjc0CjIwMDUwLDMxMjQzMzAyLjY0NDI4MTQ5CjIwMDUxLDM5OTc3MzYyLjAzOTIyMzI1CjIwMDUyLDQ0MDM1MjcyLjI5MzgzNDA3CjIwMDUzLDI1MTA5MjIxLjkyNDgxOTAyMgoyMDA1NCwyNjUwNzgzOC44MTM2ODg2NDMKMjAwNTUsMzkyNDU2MzMuMjYwNjk0NwoyMDA1NiwzMzM2MTIxMy45OTg2NjE2NjcKMjAwNTcsMjA4MTY4NDQuNDYyMzM4NzI3CjIwMDU4LDMyNTg1MjA0LjQ2NjgzMDgxMgoyMDA1OSwzMjg3MTMwNS4wNzg4MDQxODcKMjAwNjAsMjk3ODEzOTUuNTU3MTI2NTYKMjAwNjEsMTk0NTA1MjIuMDI1MjgwMzUKMjAwNjIsMzE1ODc1MTQuMjM3MTQxNTcyCjIwMDYzLDMzMDMwMDYxLjk3NjA1ODg3OAoyMDA2NCw0Nzk0ODgxOS4yNTcwODg5MwoyMDA2NSwzMTY0NTE4MC45NDUzMjI1MwoyMDA2NiwzNTMyMTMyMS45NTc2MzAwMQoyMDA2NywxOTk0MTA4NC40MzI0MDIzMgoyMDA2OCwxNTQ4OTkwOC4xNTU3NDA1NTUKMjAwNjksMjYwODQxMjQuNjU5OTgxODQ3CjIwMDcwLDQ2MjA0MzUxLjQ4NTI3OTI0CjIwMDcxLDIzNDgwMzk5LjUwNDMxNjk2CjIwMDcyLDMyNjYxNDEzLjYyMDk3NDM2NgoyMDA3MywyNTU1OTY0Ny41OTQ2Nzc1NTMKMjAwNzQsMjIxMTU4MDYuOTgwODkxODQKMjAwNzUsMjY3NDA1OTguMzI0OTgwMzEKMjAwNzYsMzEyMDcyNDQuNDQ1NDg2MTEzCjIwMDc3LDMxNjEwMDc3LjQxOTQ5MzE3CjIwMDc4LDMzNDE0NTE1LjUwMjkxMjkyNwoyMDA3OSwzMTEyMTM4Mi44NTA0NzM2ODcKMjAwODAsMjYxMDM5NTUuODE2NDM3NDEKMjAwODEsMzM5Nzk2NjguMTI3MDc2OQoyMDA4MiwzNDUxMjI1NC4yODczNjM5NgoyMDA4MywyODE1OTExMC4zODQ0MTE1CjIwMDg0LDM2NTgxOTM2LjIwNzQ2NDk2CjIwMDg1LDI5NzU4NDk1LjE5NDAzODM0CjIwMDg2LDMwMjg2MzA3LjkyODc4MDkKMjAwODcsMjY0NDU2NzYuNDcyNjcyNTk3CjIwMDg4LDM2MTY4NjM2Ljk3MDA0MDYzCjIwMDg5LDM5NjEwMzc5LjgxOTg2Njc1CjIwMDkwLDMyMjMyNDk1Ljc2NDk4ODUzNAoyMDA5MSwyOTAwOTQ3My4xOTEyMDE5MzMKMjAwOTIsMTYzMjQ0NTIuNTM2MzU3MjkxCjIwMDkzLDI1MDUyNjc1LjUwMzcwNDY0NQoyMDA5NCwxNjU3OTI2NS4xMzA5NDY5MzYKMjAwOTUsMzc4NTM5MTQuNTYzNzI4MDIKMjAwOTYsMzA2OTEyMzguMzQxMjg5MDUKMjAwOTcsMjc0NDgzMDcuOTkzNzM1NDQKMjAwOTgsMjYzODcyMTkuNDg4NzkwODQKMjAwOTksMzMxMjYwMDkuMzAxNDc3MDQ1CjIwMTAwLDI0MzUzMDY5Ljk5NzY0NzUKMjAxMDEsMzA5NDc2MjMuMzE3MjMwNTY3CjIwMTAyLDM1NzA0OTQ1LjMwMzY1MzYKMjAxMDMsMzUzMTI1MTkuNTQyOTcxNTc0CjIwMTA0LDIzNDc5MTQyLjk2OTk4NDg5MwoyMDEwNSw0MTU4MTg1Ny45MDYzMDYzODYKMjAxMDYsNDM4MzY4OTQuNTgwMDMxOTkKMjAxMDcsMjc5MjE1NjEuNzk2NDg2MgoyMDEwOCwzMTMyNDM5Ny45OTcwMzMzMgoyMDEwOSwzMTQ4OTA2MC4wODQwNzI3ODQKMjAxMTAsMjAzMDg4MzguODg2OTM2NzU0CjIwMTExLDM1NTQ2MjcyLjEyNzYzNjA1CjIwMTEyLDM4Mzk0MTUwLjQ2NzE1NjY2CjIwMTEzLDIyMDk5MDYwLjk4MDM1MTM3NAoyMDExNCwzNzkyNTUyMS4zMTI3NTYzNzQKMjAxMTUsMzcyMzgxOTIuMjE5NjM4NDIKMjAxMTYsNDY5NDE5MjAuMTk2MjkzODIKMjAxMTcsMTgzMDcwMDMuODcxMTY1NjI2CjIwMTE4LDQwOTQ2MTA4LjcyMzEwMTgKMjAxMTksMjI5OTY0MDkuMzY0OTA4MTYKMjAxMjAsNDgyMTQyMzEuMzY3ODk2NTMKMjAxMjEsMTg1Nzg5MDQuMzg5NjEyNjA4CjIwMTIyLDMxMTkzMzA4LjYzNTU1MDQwMgoyMDEyMywyOTc4MzQ2NS40MzY1NDEyMDcKMjAxMjQsMTY5MjQ2NzUuNDg2MDQ1ODA4CjIwMTI1LDUxNzYzNTE5LjQ3NDQ1NTAxNAoyMDEyNiwyNzExNDc0OC43MzQ1ODMzNjMKMjAxMjcsMjQxMTIzNjMuMTQxODM4NDU3CjIwMTI4LDE5MTU3NjgzLjIyMjkwNTA4CjIwMTI5LDMzMDY5NjcyLjQwMjUwMjc1CjIwMTMwLDM4NDQwNTMzLjA2MDQxOTE3CjIwMTMxLDIyNTk1Njg1Ljc3MjY4Mjk5NQoyMDEzMiwyMzY1MDgzOC4xNjcwNjE2NTcKMjAxMzMsMjc4ODM1NjkuMTAxNTE3OTEyCjIwMTM0LDM1MzQzNTI2LjYxODQ4NjE2NgoyMDEzNSwzMjAxMzA0My4xNzc0MzgzNwoyMDEzNiwyNTIxNTQ5OS43NzMzNjA1NAoyMDEzNywzNTg1NjEzOS45NDM3NzExNAoyMDEzOCwzMDA1Nzg4MC4zNzAxMTExOQoyMDEzOSwzNTkyMDM0Ni40MDcxNDUxOAoyMDE0MCwyNzE1OTY0NS4yOTU3NzU1MDcKMjAxNDEsMzY3MTYyOTIuMzE2MzgwNzIKMjAxNDIsMzI2Njc3NjIuOTA2NjEzNzgyCjIwMTQzLDMzMDQyOTE4LjYzNjc1MTU0NwoyMDE0NCwyMjU1OTc2OC44ODE0NzgyNDYKMjAxNDUsMjYwMjgzNDguOTc3MjMxOTk0CjIwMTQ2LDI5ODM0ODU5LjI0MTYwNDExNgoyMDE0Nyw0NDgyNTU4MC41NjM0NzA0CjIwMTQ4LDMwMDk5MDQyLjI2MjMwMTgxOAoyMDE0OSwzMDU5NDE1My40NTg5MTIwNTIKMjAxNTAsMjcyMzUwNTMuMDg0NDQ0MDE2CjIwMTUxLDM3NzE1NTMyLjgxNjI1NTM3NgoyMDE1MiwxOTA2ODY3Mi44MjUxNzUyMwoyMDE1MywyMDA4NTU1Ni4xNjgwNjkxOQoyMDE1NCwyODY4MTE2NC44NjkzNTcwNDIKMjAxNTUsMzU4NDg1ODQuNDE1MTI4NjgKMjAxNTYsMTcyNTEyMjAuNTUzNzE3MDQ3CjIwMTU3LDI5ODk1MzA4LjYwNTcwODgyMwoyMDE1OCwzNzI2OTQxOS4xNjg4NDU2NwoyMDE1OSwyNzQ4Njg4MC44OTg5MzIxNQoyMDE2MCwzMjg0NDMxMy44OTg0NjY2MDIKMjAxNjEsMzMxNzI2NDcuMDQzNTkzNjE1CjIwMTYyLDI3MDU3Njk2LjMxNTAwNjUzNgoyMDE2MywzODk4ODI4Ny45MjM4Nzc4OAoyMDE2NCw0NzM2NTA0Ni40MDE4NTM2NAoyMDE2NSwzNDIzNDIxOC41NTAxMjMwMwoyMDE2NiwzOTI1NTExOC43MzEwNzQ1MwoyMDE2NywyNTcwMjI0My4wMTYyMzg1MjYKMjAxNjgsMTc3NzQzNzAuNzM2ODk2NTkzCjIwMTY5LDM4ODg5NDMzLjkxMTE0MjYyCjIwMTcwLDI2MDY0NjYxLjAxMDM1NDYxNgoyMDE3MSw0NTg0NjYxMy4wOTIzMjMxODQKMjAxNzIsMzUyNTY4NDUuMTI2NzMwNDcKMjAxNzMsMjQwNzY2OTQuNjg4OTA4NzYzCjIwMTc0LDM1NDk4ODc3LjUxMzQyODg3NAoyMDE3NSwyODkyNjExNS4yNjc4MDM5OTMKMjAxNzYsMzUzODg4MDcuOTA3NjAxMDMKMjAxNzcsMzE1OTc5NzkuODA1ODQ3MzU4CjIwMTc4LDI0NzMwMzgyLjYxNDI1MjQ3CjIwMTc5LDIyNzU3NjI4Ljc1MTY3ODQ4CjIwMTgwLDM1MjY2OTE1LjU2NTk3Nzc0CjIwMTgxLDQ1NzMwOTUyLjg1MjgwNDMKMjAxODIsMzQwNzMyNzEuNTkzOTYxNTEKMjAxODMsMjQ2MzY2ODYuNjc5Mzk1OTQ4CjIwMTg0LDMyNTE3MjcxLjQ3OTE1NjQyNwoyMDE4NSwyODMzOTEwOC44MDY3OTEyMwoyMDE4NiwyMjk4MTI3Mi4zOTY0MDk5OAoyMDE4NywxODIyNzE5Ny42ODQzODczMwoyMDE4OCwxODg5MzEzMy45NzQ4ODE3CjIwMTg5LDM0OTk5NTc2LjYwOTg0MzE2CjIwMTkwLDQzNjA0MTIyLjcxMjAyODk2CjIwMTkxLDE3ODY5ODU3LjI0MzU5ODUxMwoyMDE5MiwzNTE4OTY2MC40NTk0Mzc1OTQKMjAxOTMsMjcxMzg0NzIuMzA4NzE3MTU0CjIwMTk0LDM3MTk3Nzk2LjUzNTA5Nzc2CjIwMTk1LDI1ODU4MzM4LjI1ODUxMTMyCjIwMTk2LDIzOTA3NjkwLjYxMTk0OTM4CjIwMTk3LDMyNjkxNjA5LjEzODQyMTA5NgoyMDE5OCwyMDk1NjY1NS4zOTY1NDQwMjQKMjAxOTksMzM5MjQ4NTYuMDcwNDQ5ODc0CjIwMjAwLDQxMTI4NjQ5Ljg3Nzk2NzMzCjIwMjAxLDMzMjQ5MTE4Ljc2NjQ3OTM3CjIwMjAyLDMzNTUxMDIwLjY0ODg4NjEzCjIwMjAzLDIzMDc3MDM5LjkzNDExNTQ0NwoyMDIwNCwyNjgyMzI5MC41NTM1MzEwMwoyMDIwNSwyMDI5MTk3NS41NTA1MjQ5NzIKMjAyMDYsNDA1MTcxODEuNzg2MzgwMTkKMjAyMDcsMzI3MTk1MjMuNDcwNDk4NjE0CjIwMjA4LDMxNTI0NTEwLjc0MjMyNTY2NAoyMDIwOSwzMTg2MDgxMS43MTUyMjMzODMKMjAyMTAsMjI1NDI4NDkuMDUzNzAxMjkKMjAyMTEsMjIxMDMzODYuMzQ5MTExODIKMjAyMTIsMzEzMjE5MjcuMzAxODI1ODQKMjAyMTMsMzgwNTAyMDEuODU2NzM0NzQKMjAyMTQsMjAyOTk4ODIuMjgwNzQ3OTA1CjIwMjE1LDIxMDUzNDU4LjQyMTgxODM5CjIwMjE2LDE3NTkwODg0LjU1NDIzMzAzCjIwMjE3LDM1OTk1MDQ2Ljg3NTA0Mjk0NQoyMDIxOCw0MzEyODIzOS4zMzg2NjA4OAoyMDIxOSw0Mjk4MzA0Ni4zMDE0NTYwMDQKMjAyMjAsMzY3NzkyNzkuNzU4Njk0Mjc2CjIwMjIxLDI1MzQ2NTQ1LjIzMzcxNzc1CjIwMjIyLDEzNjA1OTEyLjg3NjQyNDEzNgoyMDIyMywyMzA4NDQ0Ni4xOTg0NjA1MTIKMjAyMjQsMjQ1MDU1NzQuNjEyOTYyODM1CjIwMjI1LDQxNjEyNDc3Ljc1MTUzMzA5CjIwMjI2LDI5NzkxMzE0LjY5NDIwMjcwNgoyMDIyNywzNjQxMzQ4OC42MTU1MTE2ODYKMjAyMjgsMjkxMDcyMzYuNDA5MzYyNDk1CjIwMjI5LDMzNTc2Mjg5LjczNDQzNzc1NgoyMDIzMCwzMjMzNjgwMi41MjMzMTI1MQoyMDIzMSwyNjUyNTE0OS44OTA4NjU3NDMKMjAyMzIsMzcxMDY2OTQuNTI1NTQ0NDcKMjAyMzMsMzkyMjQzNzMuNTU1NDQ0NzEKMjAyMzQsMzgwOTU5NjIuNTcyOTk5NjU2CjIwMjM1LDMyNTAxNDUxLjU3MTA0NTU2CjIwMjM2LDI5NjI3MTgwLjAyMTcyMzg3OAoyMDIzNyw0MDYwMzMzNS41NjkyNTgzODQKMjAyMzgsMjk0MjUwMzQuMDEzNDMzNTc2CjIwMjM5LDI3MDk2NDYxLjY2NjAxODAyOAoyMDI0MCw0NzMxMTI0NC42Nzc4MTM2OAoyMDI0MSwxODIzMzE4Ni4wMDQ4Nzk5NgoyMDI0MiwyNTgyODA4NS44NTYzMjcwOTgKMjAyNDMsNDIwOTY3NjAuMjA1NDcyNTQ0CjIwMjQ0LDE3NjQ3ODQ4LjI2NDIzMDQyCjIwMjQ1LDMyNjI3MTUzLjA1NTIzMjA5MwoyMDI0NiwyMjk4NDgwNi42NDYzNDY4NAoyMDI0NywxNjIzMTc1Mi41MjMzODk1MzkKMjAyNDgsMzMxNzQ5NDguNDg5NDUxMjYKMjAyNDksMjI1NzQ5MzYuNTgwODUzNzI3CjIwMjUwLDM1NTA2NTcyLjE4NTQ5NDQxCjIwMjUxLDM3NzcyMzMwLjQwMzc3MDQyNAoyMDI1Miw0NjI4NDA2MS40NzAwMDU5CjIwMjUzLDIzOTMwMjYzLjM4NjQyNTQyOAoyMDI1NCwzMzQ0MDUzOC41MDMyNDY3MzIKMjAyNTUsMjUwNDQ0NTEuMzYyNzQwMTUKMjAyNTYsMjc5MzUzODAuNTc5MzAzNTk2CjIwMjU3LDE5OTUzMzE4LjgxODM2MjU1NgoyMDI1OCwzNTAxMTM3Ny4xMzExODMxNTUKMjAyNTksNDc4NzUyMzIuNDU5ODIwNjM2CjIwMjYwLDIxNjk2MzUxLjQxNzkwMTMzNwoyMDI2MSwzNTI3NTQ5Ny40NDcwMzYwOQoyMDI2MiwxNzEzMDk1NC45MTExNDMxNzYKMjAyNjMsNDIzMTUxODYuOTE4MzE1ODkKMjAyNjQsMjEzNjY3NDIuMjc4ODk5NTM2CjIwMjY1LDI3NzI5MTM4Ljc2MTQyMjMzCjIwMjY2LDI0ODU0NzQ2Ljk5NDQxOTU3CjIwMjY3LDI2NzI2Mjk2LjI2MTA2ODUwNAoyMDI2OCwyMzAzMDI0My41NTc3NjIwNgoyMDI2OSwzMTg5MDg1Ny4yOTU4MDEzMzQKMjAyNzAsMzI2MjA3ODYuOTk2NTA3NjMKMjAyNzEsNDAyNDcyNjkuOTEzNDQ4NzM2CjIwMjcyLDMwNjU2NzczLjE3NzcxMjIwMgoyMDI3MywyMzc2ODE0Ny45OTAxNDI0MDUKMjAyNzQsMjk2MzAzNzIuNzIxMzM5MTc0CjIwMjc1LDM0MzI1OTI2Ljc0OTUxMTY5CjIwMjc2LDM4NjIyODI2Ljk1MjYwOTY2CjIwMjc3LDE4NjIwMDI3LjU2NTcyNjY4MwoyMDI3OCwzNDcyODMyNC42NTI1NjYKMjAyNzksMzQzMjA2NjcuNzc5NTAzNAoyMDI4MCw0MTU5NDY4NC44NjI1MjE2MwoyMDI4MSwyNDcyMDY4MC44MzkyNzc4ODIKMjAyODIsMzM4MDkxNTQuNjU1NTc5MTA1CjIwMjgzLDIzODczODMxLjA2MDExNTE0NAoyMDI4NCwyNjg5NjMzMS4xNzMxNDMyMDgKMjAyODUsMTU2OTgzMjQuNDM2MTkxMTY0CjIwMjg2LDM0MDM4NDM2LjYzODE0NzIwNQoyMDI4NywxOTM4MzMzNi4zMDE1MzY0NjMKMjAyODgsMjU5NzUyNjcuNjA5ODk5NzkzCjIwMjg5LDI3Mzg0Njk3Ljg3OTc5NzkwMgoyMDI5MCw0Mzk3NDI3MC41MjMwMTA5CjIwMjkxLDM5OTczNDA3Ljc0MDUxMjQKMjAyOTIsMzYwMTg2NjMuNjQ5NzM5NzM1CjIwMjkzLDI3ODM4NDc2LjIwMDcyODc2CjIwMjk0LDM0NjA1Nzc0LjgyOTIxNTI2CjIwMjk1LDMxMTM2MDA0LjIwNDAxODU3NAoyMDI5NiwyNDQyMjQzMy4xNzcyNzQ3NAoyMDI5NywxNzA2OTk3Mi43MDQ4NjA4NQoyMDI5OCwyNDY0MzgyNC4zMDIwMjc5MDMKMjAyOTksMzk4MjU3NDEuNzUzMDA2MzQKMjAzMDAsMjkyOTI3OTYuMzQ4ODk3MTY3CjIwMzAxLDE4ODkwNDk5LjcxMzUwOTEyCjIwMzAyLDMzOTczNDYzLjk2Njc4MzEzNgoyMDMwMywyNDE0MjY1MC4xMzg5MjU4ODgKMjAzMDQsMjMxOTk4ODkuMTM2MDQ5NDU3CjIwMzA1LDI5OTY5NjI5LjcyMzI1NTAxCjIwMzA2LDIxNjM3MjkyLjI3MzE0MzA5OAoyMDMwNywzNjA0MzQwMC42MTQzMTk3NAoyMDMwOCwyMjA3NjM1OS4yMDE1OTY4MjMKMjAzMDksMjM3MDY1MDIuOTIzMDMyNzY4CjIwMzEwLDIwMjUwNzAyLjk5MTY1MjA3CjIwMzExLDIwNjY2NjUwLjQzOTg1MDYxMwoyMDMxMiwzNzM1MTY1Mi45NjI3MDU5NAoyMDMxMyw0MTc4NjIyOC43MDY4OTUwNAoyMDMxNCwyNDQ1MjkyMS4zMDg4NTMwNzUKMjAzMTUsMjMzNjQ2MTYuMzE5OTU5MDk3CjIwMzE2LDIxOTM2ODkyLjI2MjY5NjgwMwoyMDMxNywyMzU3NTU5NC4wNzE0MTEzOAoyMDMxOCwyNzA5OTU3OS43Mjg3OTU4MzgKMjAzMTksMjc5NjY3ODIuMjA0NTM5ODczCjIwMzIwLDE3OTQ3NTMxLjE3NDkzNjg0MgoyMDMyMSwzNzY2Nzk1Ni40NjIwOTIwMQoyMDMyMiwzNDI1MjQ5My43MDAyNTA3OQoyMDMyMywyODk2NjAwNC41ODY0MzM1MwoyMDMyNCw0MjYyMTAzNS41MjAyMjU5NAoyMDMyNSwyNDExMTk4NS41MDM4ODkyNAoyMDMyNiwzODQ0MTA3NC43NTA1OTIyOQoyMDMyNywyNTM3Njc4Ni45OTU0NjUzOTQKMjAzMjgsMTc2MTA3NDMuMjc2NzMwOTQKMjAzMjksMjA1MTg5NzguNzc3MzY1OTcKMjAzMzAsMzI2MjUzMzguMzYyMjExMTYKMjAzMzEsMjYzMDEzMjcuOTk3MDY4NjMKMjAzMzIsMzMxNTYwMjQuNzUxNTE5MzQ1CjIwMzMzLDQyNTk4ODY0LjM2MzMyNTMxCjIwMzM0LDQ3MjU5MjMzLjA4MjI4OTQKMjAzMzUsMjUzNzMxMTQuMzM4MDA5OAoyMDMzNiwyODU1Mjc5NS4yNDAzMDcwMzcKMjAzMzcsMzYyODU2NDkuNDkyMTYzMTk2CjIwMzM4LDIwOTY4NjEyLjk3MzY4OTQ5MwoyMDMzOSw0ODM4NjkwMS40NzQzMjY5OAoyMDM0MCwzNjg0NDQ0OS44ODQxMzU2OQoyMDM0MSwzODc5Nzg4MS44NTgzMTk0MgoyMDM0MiwxODMyNjE4Mi4yMjUwOTIzMzcKMjAzNDMsMzU4OTQyNjguOTQzODUwODc1CjIwMzQ0LDI3MzA5ODcxLjQwODUwOTc1NAoyMDM0NSwxODA4NzUyNC4yOTY5MTg3MjcKMjAzNDYsNDA0ODgyMDkuOTQxMTcxNzk1CjIwMzQ3LDM5MTMzNjIxLjgxNjk1NjkKMjAzNDgsMzU0Njc0OTguNjU5MTM1NDQ2CjIwMzQ5LDQyNTY5MTQxLjU5NTQ0NzkKMjAzNTAsMjM0NTgxNTYuOTEyNDY1NTM1CjIwMzUxLDMwMTAwMTQ0LjU1ODIzMTY3NAoyMDM1MiwyNDQ4MTYxMi4xNzgzMzcwNjcKMjAzNTMsMzU2MDk0NDQuOTEyMDIxOAoyMDM1NCwyNDg0MjM0Ny41MTA4NDU2NTcKMjAzNTUsMjI5NDk0MDUuNTgwMzU2OTIKMjAzNTYsMjM0OTI5MDcuMjU0MTA0NzkKMjAzNTcsNDg2NTU0NzAuNjQ1NDI2NjQ2CjIwMzU4LDQwNzU1ODA3Ljg2MDM3MjUzNgoyMDM1OSwzMzc0MTgzNy4zNDI0MjA1MjYKMjAzNjAsMjgyMjU1MTMuODYwMjY5NTQ3CjIwMzYxLDI2NDI2MDk2LjEyNTkzNDU0CjIwMzYyLDMwMTQwODkwLjU3OTQyNTQzCjIwMzYzLDI2MzE0NDY0Ljg2NTM3NzQyMgoyMDM2NCwzNzAwNTExMS4zNjQ2MzQyNAoyMDM2NSwxODYxNDA5Ny43NzY3MDgxMQoyMDM2NiwyNjY4NDA5NS41NTY1Nzk4ODQKMjAzNjcsMjI4MDcxMDAuOTczODk2MDg2CjIwMzY4LDI2NTgwMzU1LjAzMTMzMTk2CjIwMzY5LDQ3MDUwNjQ3LjQ3NDY5MDU0CjIwMzcwLDM1NzUxMjc1LjE0NzU2MTcxNAoyMDM3MSw1MDAwNzI2NC4yNzgwNzA0MzUKMjAzNzIsMzk2NzE5MDIuMjk4ODk2OTYKMjAzNzMsMzE5MjYyODAuNjAzODA3ODk2CjIwMzc0LDIyNTQzMzMyLjcwMDk1NjM4NgoyMDM3NSwyOTIyMjAwMi44MzY3OTA3OTMKMjAzNzYsMjA1MjQzNDAuMjE3NzY2MjM3CjIwMzc3LDMyMDc5ODIxLjkxOTMxNTkxNgoyMDM3OCwzOTE0OTY5NC42MzkwOTQ1NwoyMDM3OSwyOTUyODEwMC44ODI5NTA5NTQKMjAzODAsMjcyNjQ3OTUuNzM4MTQyNDMKMjAzODEsNDcxNDY1OTMuMzIwODgwNDgKMjAzODIsMjE2NDAwNjAuMjQ2MDkwNzQKMjAzODMsMzM2MTY0NzUuMzY4MTYyNjMKMjAzODQsNTUzOTYwOTcuNDkwMTQ0MTcKMjAzODUsMTk1NDcyMTQuMzIyMzczMzgKMjAzODYsMjkzOTYzNzYuNjE3NDk5NzgKMjAzODcsMjM3NTIyMjQuNDQ5MDE4OTMzCjIwMzg4LDE4OTgzMjIzLjA2MTkwMjcxNwoyMDM4OSwzMDE0MzI2Mi4wMDgyNTI1OQoyMDM5MCw2MDg1MDE1My40MDY0OTI3MgoyMDM5MSwxOTkyNjIwOC44Njc5MzQ2NDQKMjAzOTIsNDYyMjkwODcuMzk3ODE2MjI2CjIwMzkzLDIzNDI1MjM1LjY2MTczMDU4NAoyMDM5NCwxNDgzOTYyNC43MDIzNDMzNjcKMjAzOTUsMzk2MjQ4MjguMDMwODI5NjcKMjAzOTYsMjk1NDc3MDEuMTY4MzQxODMKMjAzOTcsMzE5OTU5ODUuMTQzNDE4MDQ4CjIwMzk4LDM0NzYxNDAxLjY4NTkzNzE5NgoyMDM5OSwzMDA3NzI2Ny4yNjkzNzYyNQoyMDQwMCwyMjUwMDc4NC40NzIzOTYxOAoyMDQwMSwzMDU0NDQ3MS42NjU0MTg0MgoyMDQwMiwzMDIyNDY4OS4yMDcwNjk4MDcKMjA0MDMsMzUwNzAyNDguOTMyMDI5MTYKMjA0MDQsMjEwOTUwMTMuNjExMDE4ODQ0CjIwNDA1LDM3Nzg3ODE3LjUyNDMzMDYxNgoyMDQwNiw0NjcyNTk4Ni4yNTkyOTk3NwoyMDQwNywxNzEzMjIwMi4yODQ4MzQ5OTYKMjA0MDgsMjI4MjY2MTMuMTc4MDY4MzAzCjIwNDA5LDMzMjA3NTEzLjY3MDUwNzUwNgoyMDQxMCwxOTMxODMxNS4yNTY0NzMxMzUKMjA0MTEsMjIwNTg0MjYuMzg4ODc2NTI4CjIwNDEyLDMxOTM2NjExLjYwMzE4NDgyCjIwNDEzLDIzMDA2NjA4LjYxMTA3MDIxNgoyMDQxNCwyNjA0MjA4NS4yOTg3ODI5NjcKMjA0MTUsMjMzNDIyODYuMjAxMTUwODEyCjIwNDE2LDMwNzg0Njc3LjA2ODQyMjM5MgoyMDQxNywyNjI3MDQ4My4zNjExNTc5NTQKMjA0MTgsMjcwNDI2OTAuNjg4OTE2Nzg4CjIwNDE5LDE5Mzg2NDc2LjQxMjYwMDEyMgoyMDQyMCwzMjQ1MDU1NC4zMjkwNTMyOTgKMjA0MjEsMzA3MjkyNDEuMjUzMzc3OTcKMjA0MjIsMzc0MjM4NTIuNjU3MDc1NDM1CjIwNDIzLDM0NTEzOTI4Ljc3MjI3NjU4CjIwNDI0LDMyNjk2NjI5LjY3NzcyNzU3NgoyMDQyNSwyMDMwNjMyNC40OTk4MTU3NTUKMjA0MjYsMjk2MjczMTUuMzA5NDU5NDMzCjIwNDI3LDI1NDI2OTAzLjUyMDUwOTY2OAoyMDQyOCwzNzY1MTkzMi40MTQ5MTkwNAoyMDQyOSwzMTg2OTE1MC4yNDUyNDQ0CjIwNDMwLDE3OTAxNTkzLjUwODA0MDE1NgoyMDQzMSwyNTUwMDU0Mi4xMTI2OTE0MgoyMDQzMiwyMzM3NjIyMC44NDMyMzAwNzYKMjA0MzMsMzkwOTc1MjEuODI5MzcyODc1CjIwNDM0LDIwMTk3ODYwLjA2NTMwNzc1CjIwNDM1LDMyMjU3NDEzLjIwMzc2NTIzMgoyMDQzNiw0MzI1NDEwOS45MDk1NjA1MgoyMDQzNywzODc0MzAyNy4xMDU2NDAxMQoyMDQzOCw0MTk4NjY3NS41MDQ4NjI1NQoyMDQzOSwzODUyOTEyOS43MDQwODA0MgoyMDQ0MCwyMTIzNzE0My43MzIxNjY5NzYKMjA0NDEsMjQyNTgxNDUuNjgxOTM5MTIKMjA0NDIsMzAxNzMzMTguNzE1NjQwOTE3CjIwNDQzLDE4NzczMTE0LjY1MTY2MDc1CjIwNDQ0LDMwNDU1NjgwLjIyNjI3NzY2OAoyMDQ0NSwyNDE4NDU3MS45MDcyOTAzMTMKMjA0NDYsMjU1NTQzOTEuNTY5NTUyOTA2CjIwNDQ3LDE5Mzk3NDAyLjY5MDIyMzEyOAoyMDQ0OCwyMDI0MDU3NS44NDQxNTIyMjMKMjA0NDksNDQ0MzQzNjQuODYxNDEwMjc1CjIwNDUwLDMxNDg5NjQ3LjY1NTQwMzc4CjIwNDUxLDM1OTkwODM2Ljg3NTQxMDM1NgoyMDQ1MiwxODgxMzcyNS45NTczNzM3MjMKMjA0NTMsMjE5NTQ4MTYuNzI0NTQ3NjU0CjIwNDU0LDI5MTk3NzYyLjkyNTQwMDk3CjIwNDU1LDQ2Njg5NDgzLjE0OTI1NTY4NgoyMDQ1NiwxNDk2MzU3My4yMDE2NTA4OAoyMDQ1Nyw0NjUzNTkwNC43MjkwNDQxODQKMjA0NTgsMjg5MzU1MDIuNDg1MDQ1NDg1CjIwNDU5LDI4MjM0Mzc4Ljc0Mzg4NDYxCjIwNDYwLDI3ODc3MTMyLjk2MDY1ODE3OAoyMDQ2MSwzOTgyNjE2MS4xMzg2MDUxODUKMjA0NjIsMTgwOTI2MzEuODI4NTcwNjk0CjIwNDYzLDM3NzM0MzE3LjE1MzA1MTYxNQoyMDQ2NCwxOTc4MjM5Ny45Njg4Mzk4CjIwNDY1LDQxMTY5NDczLjAxNjA2OTU5CjIwNDY2LDQxNTcxMDg3LjU0NTYzMjExCjIwNDY3LDE2MjE0NjgxLjE3MTEyNjc1MQoyMDQ2OCwxNzIxNjMzMC45NjY2MDI5MzYKMjA0NjksNDA4NzYzODcuNjMwNDg3NTU0CjIwNDcwLDI4NTY5MDM5Ljc1OTQyMjY2CjIwNDcxLDI3OTc1NTU3LjYwNDg5Nzc2CjIwNDcyLDM0OTg1NzczLjY1MjIxOTI5NgoyMDQ3Myw0MTU4MDc3OC4yNjc0NDAxMjUKMjA0NzQsMTk1MTk2MTAuMTc4MzE0NTYKMjA0NzUsMzM2MDY3NTAuNDg3ODUyMzQKMjA0NzYsMTI1ODkyMTkuMDcyNzA3MDE0CjIwNDc3LDI5NjAzODkxLjA3MTgzNjY4CjIwNDc4LDM2ODk3NTIxLjEzMDY2NDUyCjIwNDc5LDQyNTUxNTc0LjE0OTU0MzkyNgoyMDQ4MCwzMzMxNzMyOS44OTA2MDY4MzUKMjA0ODEsMzczODIyMzcuNDI3NDI1NzkKMjA0ODIsMzQ3NTcwMzUuNDkxNzQ3MjgKMjA0ODMsMzQ4ODMxMjAuNTAyNjY2MjkKMjA0ODQsNDk3MTk0NzUuNTc0OTgwODUKMjA0ODUsMjg3NzI4NTkuMDM5MzU0NDUKMjA0ODYsMTU0MzQwNzIuMjM0NjQ0MTMyCjIwNDg3LDE4MTkzODY0LjgxMzY5NjE1MwoyMDQ4OCwyOTgxODcwNS44OTkzNzUwNTUKMjA0ODksMjYxOTA4OTAuMDU2NDYxODgKMjA0OTAsMjQ5NTE3ODcuNDQ3MjAxNjgKMjA0OTEsMzExMjgzMjEuODk0MzYwNDMKMjA0OTIsMjkzNzY5OTMuNzQ1MjAzMjgzCjIwNDkzLDI3MDEwNjA5LjY5ODgzNTQ2NgoyMDQ5NCwyNzcxODczNS44NDAxOTk1CjIwNDk1LDMxNDg3NDEzLjg0OTYyMjE3NQoyMDQ5NiwzMjU2NTMyMC41MDQ2NjIzNDIKMjA0OTcsMzAwNTEyODYuNDQxMzk2NjE2CjIwNDk4LDM0MDQ5OTEyLjUxNjk0MDQyCjIwNDk5LDQyODg2MzE3LjU4MDc5OTgKMjA1MDAsMjc0OTI0MTEuNjc0MjI1MDg0Cg==","scored_v7_backup/submission.ipynb":"..."}')
FILE_HASHES = {'legacy_reference.py': '1a24f9f116170ea54bd3242264d5cd2a150c3429e039a50cb11c3112e52247c1', 'legacy_models.py': '031c25ba5a60cf3e6e4c8247553ee951ff71742c846eb103246140929f6cdfba', 'baseline.py': '0513655049b5d7c0e4e57b3339596e718779cd7bcbcc49bafd76e14e5bd9dad1', 'modeling.py': '44177560fc1ed64744662ba3159acff5026644ca74423f52cac6bcc3966eefb3', 'engine.py': '9354369db39326d90534fd2542657005ca22b8b8680eec0e619902e6af6e4610', 'workflow.py': '71450094d272322589ea7bf638768bb3f04c2e2c3484948c77e3988b554200e1', 'reference_manifest.json': 'be46ba7d91b5c583570f4ceaf4919c292d4fba460dec58998873084638ae2c78', 'v5_contract.json': '3446412f01d536a31123142b984392216efd29094965500251386eac1814ab69', 'baseline_snapshots/v7_42.npz': '48c19dded9e46b1b5a0760ce42abb5cad33d62cc3300f92958bdf6a727026f29', 'baseline_snapshots/v7_71.npz': 'd2dc4282cfc3d6771c7e1ffd62629d24f204da19d975e430190e6640774c8b06', 'baseline_snapshots/v7_city.npz': '06f6ed4e891869b46c0e84b76a703714e5e73fbabd4e2de64d1f596515e7d585', 'baseline_snapshots/v7_strat_27421.npz': 'e504f31f2a1f260cffed4900890c1849bcfd42cf69be9ee697689d93adb66b2a', 'scored_v7_backup/submission.csv': '5f20ea8d610e325c0eae4727b7ed8c737ce2d881239792c9a6af5ce4e3dec1ba', 'scored_v7_backup/submission.ipynb': 'b0723c81bf424dc8df2cf06778e46586b82d3ee0bf731255cd2f2f05c1dbb078'}
for relative, encoded in BUNDLE.items():
    path = WORK_ROOT / relative
    value = base64.b64decode(encoded)
    assert hashlib.sha256(value).hexdigest() == FILE_HASHES[relative]
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        assert path.read_bytes() == value, 'Runtime source changed; choose fresh output storage.'
    else:
        path.write_bytes(value)
sys.path.insert(0, str(WORK_ROOT))
from engine import Engine
import workflow
e = Engine(DATA_DIR, WORK_ROOT / 'results', budget_hours=18)
e.cache.stage = 'audit'
workflow.audit(e)
print('PASS: input, implementation, snapshot and scored-file hashes.')


## Screen

Freshly fit sixteen settings on five folds of assignments 42 and 71.

In [ ]:
e.cache.stage = 'screen'
workflow.screen(e)


## Refine

Refine only settings improving on both assignments, then freeze.

In [ ]:
e.cache.stage = 'refine'
workflow.refine(e)


## Confirm

Confirm only frozen finalists. No finalist means no fabricated confirmation score.

In [ ]:
e.cache.stage = 'confirm'
workflow.confirm(e)


## Export

Freshly reproduce scored V7, then export qualified candidates or its exact fallback.

In [ ]:
e.cache.stage = 'export'
workflow.export(e)


## Measured results

Positive gains mean lower RMSE than V7. These figures are recomputed from this execution. Cross-validation scores and portal scores describe different datasets.

In [ ]:
table = pd.read_csv(e.out / 'screening.csv')
best = table.groupby('family', sort=False).head(1)
print(best[['name', 'weight', 'gain_42', 'gain_71', 'mean_rmse']].to_string(index=False))
decision = json.loads((e.out / 'decision.json').read_text())
print('Decision:', decision['decision'])
reproduction = json.loads((e.out / 'v7_full_reproduction.json').read_text())
assert reproduction['maximum_difference_INR'] < .01
if not REUSE_CHECKPOINTS:
    assert e.cache.reads == 0
print('V7 maximum reproduction difference (INR):', reproduction['maximum_difference_INR'])
print('Checkpoint reads:', e.cache.reads)
try:
    import matplotlib.pyplot as plt
except ImportError:
    pass
else:
    fig, ax = plt.subplots(figsize=(8, 4.2))
    pos = np.arange(len(best))
    ax.barh(pos - .17, best['gain_42']/1000, height=.3, label='42', color='#2563eb')
    ax.barh(pos + .17, best['gain_71']/1000, height=.3, label='71', color='#0f766e')
    ax.set_yticks(pos, best['family'])
    ax.axvline(0, color='#334155', linewidth=1)
    ax.set(xlabel='RMSE improvement over V7 (INR thousands)', title='Best screened blend in each family')
    ax.legend(title='Assignment')
    fig.tight_layout()
    plt.show()


## Files

The fallback is byte-identical to the already scored V7. Preserve its matching notebook. The research archive includes original sources, provenance and measured results. It is not a portal CSV. No Olympus or one-time coding upload is performed.

In [ ]:
import zipfile
archive = OUTPUT_DIR / 'V8_research_results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in WORK_ROOT.rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts:
            z.write(path, path.relative_to(WORK_ROOT))
print('Saved:', archive)
if not decision['trials']:
    csv_path = WORK_ROOT / 'recommended_submission' / 'submission.csv'
    assert hashlib.sha256(csv_path.read_bytes()).hexdigest() == e.manifest['preserved_v7'][csv_path.name]
    print('Keep V7; CSV hash:', hashlib.sha256(csv_path.read_bytes()).hexdigest())
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(archive))
